In [9]:
# =========================================================
# TRAVEL AGENT — PHASE 9
# PRODUCTION-READY RECOMMENDATION ENGINE
# =========================================================
#
# This notebook starts from the validated artifacts produced
# by the previous model-building and evaluation phases.
#
# IMPORTANT:
# We do NOT depend on variables from previous notebooks.
# Everything required is loaded explicitly from disk.
# =========================================================


from pathlib import Path

import json
import pickle

import numpy as np
import pandas as pd


print("=" * 60)
print("TRAVEL AGENT — PHASE 9")
print("PRODUCTION-READY RECOMMENDATION ENGINE")
print("=" * 60)


# ---------------------------------------------------------
# Determine project root.
#
# This notebook is inside:
#
#     Travel_Agent/notebooks/
#
# Therefore ".." refers to the project root.
# ---------------------------------------------------------

PROJECT_ROOT = Path("..").resolve()


# ---------------------------------------------------------
# Define important directories.
# ---------------------------------------------------------

MODEL_DIR = (
    PROJECT_ROOT / "models"
)

HANDOFF_DIR = (
    MODEL_DIR / "recommendation_engine"
)


print("\nProject root:")
print(PROJECT_ROOT)

print("\nModel directory:")
print(MODEL_DIR)

print("\nRecommendation artifact directory:")
print(HANDOFF_DIR)

TRAVEL AGENT — PHASE 9
PRODUCTION-READY RECOMMENDATION ENGINE

Project root:
C:\Users\Rushi\Desktop\Travel_Agent

Model directory:
C:\Users\Rushi\Desktop\Travel_Agent\models

Recommendation artifact directory:
C:\Users\Rushi\Desktop\Travel_Agent\models\recommendation_engine


In [10]:
# =========================================================
# LOAD MASTER ARTIFACT MANIFEST
# =========================================================
#
# The manifest tells us:
#   - which artifacts were saved
#   - destination count
#   - model dimensions
#   - validation status
#   - configuration
#
# This acts as the contract between the previous notebook
# and this new production notebook.
# =========================================================


manifest_path = (
    HANDOFF_DIR / "artifact_manifest.json"
)


# ---------------------------------------------------------
# Verify that the manifest exists.
# ---------------------------------------------------------

if not manifest_path.exists():

    raise FileNotFoundError(
        "Artifact manifest not found:\n"
        f"{manifest_path}"
    )


# ---------------------------------------------------------
# Load manifest.
# ---------------------------------------------------------

with open(
    manifest_path,
    "r",
    encoding="utf-8"
) as file:

    artifact_manifest = json.load(
        file
    )


print("=" * 60)
print("ARTIFACT MANIFEST LOADED")
print("=" * 60)


print(
    "\nProject:",
    artifact_manifest["project"]
)

print(
    "Artifact version:",
    artifact_manifest["artifact_version"]
)

print(
    "Status:",
    artifact_manifest["status"]
)

print(
    "Destination count:",
    artifact_manifest["destination_count"]
)

print(
    "Evaluation complete:",
    artifact_manifest["evaluation_complete"]
)

ARTIFACT MANIFEST LOADED

Project: Travel Agent
Artifact version: recommendation_engine_v1
Status: validated
Destination count: 50
Evaluation complete: True


In [11]:
# =========================================================
# VALIDATE ARTIFACT HANDOFF
# =========================================================
#
# We fail immediately if the previous phase did not produce
# the expected artifacts.
#
# This is much safer than discovering missing files halfway
# through recommendation generation.
# =========================================================


print("=" * 60)
print("VALIDATING ARTIFACT HANDOFF")
print("=" * 60)


# ---------------------------------------------------------
# Expected core model files.
# ---------------------------------------------------------

required_files = [

    MODEL_DIR / "feature_scaler.pkl",

    MODEL_DIR / "pca_model.pkl",

    HANDOFF_DIR / "processed_df.pkl",

    HANDOFF_DIR / "normalized_features.pkl",

    HANDOFF_DIR / "destination_profile_scores.pkl",

    HANDOFF_DIR / "destination_similarity.pkl",

    HANDOFF_DIR / "destination_order.pkl",

    HANDOFF_DIR / "availability.pkl",

    HANDOFF_DIR / "recommendation_config.json",

    HANDOFF_DIR / "feature_metadata.json",

    HANDOFF_DIR / "artifact_manifest.json"
]


# ---------------------------------------------------------
# Check every required file.
# ---------------------------------------------------------

missing_files = []

for file_path in required_files:

    if not file_path.exists():

        missing_files.append(
            str(file_path)
        )


if missing_files:

    print("\n✗ Missing files:")

    for file_path in missing_files:

        print(
            "  -",
            file_path
        )


    raise FileNotFoundError(
        "Artifact handoff validation failed."
    )


print(
    "\n✓ All required artifact files exist."
)


# ---------------------------------------------------------
# Validate destination count from the manifest.
# ---------------------------------------------------------

if (
    artifact_manifest["destination_count"]
    != 50
):

    raise ValueError(
        "Manifest destination count is not 50."
    )


# ---------------------------------------------------------
# Validate that the previous phase was marked as complete.
# ---------------------------------------------------------

if not artifact_manifest[
    "evaluation_complete"
]:

    raise ValueError(
        "Previous model evaluation is not marked complete."
    )


print(
    "✓ Destination count validated: 50"
)

print(
    "✓ Previous evaluation marked complete"
)

print(
    "\n✓ ARTIFACT HANDOFF VALIDATION PASSED"
)

VALIDATING ARTIFACT HANDOFF

✓ All required artifact files exist.
✓ Destination count validated: 50
✓ Previous evaluation marked complete

✓ ARTIFACT HANDOFF VALIDATION PASSED


In [12]:
# =========================================================
# LOAD CORE ML ARTIFACTS
# =========================================================
#
# These are the fitted objects created during model building.
#
# We load them instead of fitting anything again.
# =========================================================


print("=" * 60)
print("LOADING CORE ML ARTIFACTS")
print("=" * 60)


# ---------------------------------------------------------
# Load feature scaler.
# ---------------------------------------------------------

with open(
    MODEL_DIR / "feature_scaler.pkl",
    "rb"
) as file:

    feature_scaler = pickle.load(
        file
    )


print(
    "✓ Feature scaler loaded"
)


# ---------------------------------------------------------
# Load PCA model.
# ---------------------------------------------------------

with open(
    MODEL_DIR / "pca_model.pkl",
    "rb"
) as file:

    pca_model = pickle.load(
        file
    )


print(
    "✓ PCA model loaded"
)


# ---------------------------------------------------------
# Display model information.
# ---------------------------------------------------------

print(
    "\nScaler type:",
    type(feature_scaler)
)

print(
    "PCA type:",
    type(pca_model)
)

print(
    "PCA components:",
    getattr(
        pca_model,
        "n_components_",
        "unknown"
    )
)


print(
    "\n✓ CORE ML ARTIFACTS LOADED"
)

LOADING CORE ML ARTIFACTS
✓ Feature scaler loaded
✓ PCA model loaded

Scaler type: <class 'sklearn.preprocessing._data.StandardScaler'>
PCA type: <class 'sklearn.decomposition._pca.PCA'>
PCA components: 15

✓ CORE ML ARTIFACTS LOADED


In [13]:
# =========================================================
# LOAD RECOMMENDATION ENGINE ARTIFACTS
# =========================================================
#
# Everything below is loaded independently.
#
# This means the recommendation engine will not depend on
# variables from Notebook 08.
# =========================================================


print("=" * 60)
print("LOADING RECOMMENDATION ARTIFACTS")
print("=" * 60)


# ---------------------------------------------------------
# Helper function for loading Pickle files.
# ---------------------------------------------------------

def load_pickle(
    path,
    description
):
    """
    Load a pickle artifact and verify that loading succeeded.
    """

    with open(
        path,
        "rb"
    ) as file:

        obj = pickle.load(
            file
        )


    print(
        f"✓ {description}"
    )

    return obj


# ---------------------------------------------------------
# Load processed dataset.
# ---------------------------------------------------------

processed_df = load_pickle(
    HANDOFF_DIR / "processed_df.pkl",
    "Processed dataset"
)


# ---------------------------------------------------------
# Load normalized preference features.
# ---------------------------------------------------------

normalized_features = load_pickle(
    HANDOFF_DIR / "normalized_features.pkl",
    "Normalized preference features"
)


# ---------------------------------------------------------
# Load destination interest profiles.
# ---------------------------------------------------------

destination_profile_scores = load_pickle(
    HANDOFF_DIR / "destination_profile_scores.pkl",
    "Destination interest profiles"
)


# ---------------------------------------------------------
# Load similarity matrix.
# ---------------------------------------------------------

destination_similarity = load_pickle(
    HANDOFF_DIR / "destination_similarity.pkl",
    "Destination similarity matrix"
)


# ---------------------------------------------------------
# Load destination ordering.
# ---------------------------------------------------------

destination_order_df = load_pickle(
    HANDOFF_DIR / "destination_order.pkl",
    "Destination ordering"
)


# ---------------------------------------------------------
# Load availability information.
# ---------------------------------------------------------

availability_df = load_pickle(
    HANDOFF_DIR / "availability.pkl",
    "Availability data"
)


print(
    "\n✓ ALL RECOMMENDATION ARTIFACTS LOADED"
)

LOADING RECOMMENDATION ARTIFACTS
✓ Processed dataset
✓ Normalized preference features
✓ Destination interest profiles
✓ Destination similarity matrix
✓ Destination ordering
✓ Availability data

✓ ALL RECOMMENDATION ARTIFACTS LOADED


In [14]:
# =========================================================
# LOAD RECOMMENDATION CONFIGURATION
# =========================================================
#
# The configuration contains the exact weights and feature
# definitions used by the validated recommendation engine.
# =========================================================


config_path = (
    HANDOFF_DIR /
    "recommendation_config.json"
)


with open(
    config_path,
    "r",
    encoding="utf-8"
) as file:

    recommendation_config = json.load(
        file
    )


print("=" * 60)
print("RECOMMENDATION CONFIGURATION")
print("=" * 60)


print(
    json.dumps(
        recommendation_config,
        indent=4
    )
)

RECOMMENDATION CONFIGURATION
{
    "preference_groups": {
        "budget": [
            "avg_flight_price",
            "min_hotel_price",
            "avg_hotel_price",
            "max_hotel_price"
        ],
        "flight": [
            "flight_count",
            "avg_flight_price",
            "avg_total_duration",
            "avg_outbound_stops",
            "avg_return_stops"
        ],
        "accommodation": [
            "hotel_count",
            "room_count",
            "min_hotel_price",
            "avg_hotel_price",
            "max_hotel_price",
            "avg_allotment"
        ],
        "weather": [
            "temperature",
            "feels_like",
            "humidity",
            "wind_speed",
            "cloudiness",
            "visibility",
            "rain_1h"
        ],
        "destination_characteristics": [
            "sight_count",
            "park_count",
            "restaurant_count",
            "water_count",
            "forest_cou

In [15]:
# =========================================================
# LOAD FEATURE METADATA
# =========================================================
#
# This keeps the feature definitions independent from the
# previous notebook.
# =========================================================


metadata_path = (
    HANDOFF_DIR /
    "feature_metadata.json"
)


with open(
    metadata_path,
    "r",
    encoding="utf-8"
) as file:

    feature_metadata = json.load(
        file
    )


print("=" * 60)
print("FEATURE METADATA LOADED")
print("=" * 60)


print(
    "Preference groups:"
)

for group in feature_metadata[
    "preference_groups"
]:

    print(
        "  -",
        group
    )


print(
    "\nInterest profiles:"
)

for profile in feature_metadata[
    "interest_profiles"
]:

    print(
        "  -",
        profile
    )


print(
    "\nDestination count:",
    feature_metadata[
        "destination_count"
    ]
)


print(
    "\n✓ FEATURE METADATA LOADED"
)

FEATURE METADATA LOADED
Preference groups:
  - budget
  - flight
  - accommodation
  - weather
  - destination_characteristics

Interest profiles:
  - nature
  - sightseeing
  - water_coastal
  - wildlife

Destination count: 50

✓ FEATURE METADATA LOADED


In [16]:
# =========================================================
# DESTINATION ALIGNMENT & ARTIFACT INTEGRITY VALIDATION
# =========================================================
#
# This validation is intentionally strict.
#
# Every recommendation artifact must refer to the exact same
# 50 destinations in the exact same order.
# =========================================================


print("=" * 60)
print("DESTINATION ALIGNMENT VALIDATION")
print("=" * 60)


# ---------------------------------------------------------
# Extract the trusted destination order.
# ---------------------------------------------------------

destinations = (

    destination_order_df[
        "destination"
    ]
    .astype(str)
    .tolist()
)


# ---------------------------------------------------------
# Basic destination validation.
# ---------------------------------------------------------

if len(destinations) != 50:

    raise ValueError(
        f"Expected 50 destinations, "
        f"found {len(destinations)}."
    )


if len(set(destinations)) != 50:

    raise ValueError(
        "Duplicate destinations detected."
    )


print(
    "Destination count:",
    len(destinations)
)

print(
    "Unique destinations:",
    len(set(destinations))
)


# ---------------------------------------------------------
# Processed dataset validation.
# ---------------------------------------------------------

processed_destinations = (

    processed_df[
        "destination"
    ]
    .astype(str)
    .tolist()
)


if processed_destinations != destinations:

    raise ValueError(
        "Processed dataset destination order "
        "does not match destination_order.pkl."
    )


print(
    "✓ Processed dataset alignment"
)


# ---------------------------------------------------------
# Interest profile validation.
# ---------------------------------------------------------

profile_destinations = (

    destination_profile_scores[
        "destination"
    ]
    .astype(str)
    .tolist()
)


if profile_destinations != destinations:

    raise ValueError(
        "Interest profile destination order "
        "does not match destination order."
    )


print(
    "✓ Interest profile alignment"
)


# ---------------------------------------------------------
# Availability validation.
# ---------------------------------------------------------

availability_destinations = (

    availability_df[
        "destination"
    ]
    .astype(str)
    .tolist()
)


if availability_destinations != destinations:

    raise ValueError(
        "Availability destination order "
        "does not match destination order."
    )


print(
    "✓ Availability alignment"
)


# ---------------------------------------------------------
# Similarity matrix validation.
# ---------------------------------------------------------

if (
    list(
        destination_similarity.index.astype(str)
    )
    !=
    destinations
):

    raise ValueError(
        "Similarity matrix row order "
        "does not match destination order."
    )


if (
    list(
        destination_similarity.columns.astype(str)
    )
    !=
    destinations
):

    raise ValueError(
        "Similarity matrix column order "
        "does not match destination order."
    )


print(
    "✓ Similarity matrix alignment"
)


# ---------------------------------------------------------
# Feature dataset row-count validation.
# ---------------------------------------------------------

if len(normalized_features) != 50:

    raise ValueError(
        "Normalized features do not contain 50 rows."
    )


# ---------------------------------------------------------
# Final summary.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "✓ DESTINATION ALIGNMENT VALIDATION PASSED"
)

print(
    "=" * 60
)

DESTINATION ALIGNMENT VALIDATION
Destination count: 50
Unique destinations: 50
✓ Processed dataset alignment
✓ Interest profile alignment
✓ Availability alignment
✓ Similarity matrix alignment

✓ DESTINATION ALIGNMENT VALIDATION PASSED


In [17]:
# =========================================================
# PHASE 9A — RECOMMENDATION SCORING
# CELL 9 — FEATURE GROUP DEFINITIONS
# =========================================================
#
# These groups are loaded from the saved configuration rather
# than being recreated manually.
#
# This guarantees that the production engine uses the same
# feature definitions that were validated previously.
# =========================================================


# ---------------------------------------------------------
# Extract preference groups from the saved configuration.
# ---------------------------------------------------------

preference_groups = (
    recommendation_config[
        "preference_groups"
    ]
)


# ---------------------------------------------------------
# Extract the direction of each feature.
#
# Example:
#
#   avg_hotel_price -> lower is better
#   hotel_count     -> higher is better
# ---------------------------------------------------------

preference_directions = (
    recommendation_config[
        "preference_directions"
    ]
)


# ---------------------------------------------------------
# Extract interest profile names.
# ---------------------------------------------------------

interest_profiles = (
    recommendation_config[
        "interest_profiles"
    ]
)


# ---------------------------------------------------------
# Extract the scoring weights.
# ---------------------------------------------------------

travel_factor_weight = float(
    recommendation_config[
        "travel_factor_weight"
    ]
)

interest_factor_weight = float(
    recommendation_config[
        "interest_weight"
    ]
)

hybrid_preference_weight = float(
    recommendation_config[
        "hybrid_preference_weight"
    ]
)

hybrid_similarity_weight = float(
    recommendation_config[
        "hybrid_similarity_weight"
    ]
)


print("=" * 60)
print("RECOMMENDATION CONFIGURATION READY")
print("=" * 60)


print(
    "\nPreference groups:",
    list(preference_groups.keys())
)

print(
    "\nInterest profiles:",
    interest_profiles
)

print(
    "\nTravel factor weight:",
    travel_factor_weight
)

print(
    "Interest factor weight:",
    interest_factor_weight
)

print(
    "Hybrid preference weight:",
    hybrid_preference_weight
)

print(
    "Hybrid similarity weight:",
    hybrid_similarity_weight
)

RECOMMENDATION CONFIGURATION READY

Preference groups: ['budget', 'flight', 'accommodation', 'weather', 'destination_characteristics']

Interest profiles: ['nature', 'sightseeing', 'water_coastal', 'wildlife']

Travel factor weight: 0.6
Interest factor weight: 0.4
Hybrid preference weight: 0.7
Hybrid similarity weight: 0.3


In [25]:
# =========================================================
# CELL 10 — FIXED ZERO-SAFE PREFERENCE SCORING
# =========================================================
#
# IMPORTANT FIX:
#
# normalized_features was saved with a RangeIndex:
#
#     0, 1, 2, ..., 49
#
# while the recommendation engine uses destination names:
#
#     Agra, Ahmedabad, ..., Wayanad
#
# We explicitly assign the trusted destination names to the
# normalized feature rows BEFORE calculating group scores.
#
# This prevents Pandas from incorrectly aligning RangeIndex
# with destination-name indexes.
# =========================================================


def calculate_preference_score(
    user_preferences
):
    """
    Calculate personalized travel-preference scores.

    Returns one finite score for each of the 50 destinations.

    The function is safe when:
        - all preference weights are zero
        - some groups are inactive
        - some feature values are missing
        - some external data is unavailable
    """

    # -----------------------------------------------------
    # STEP 1 — Create a destination-aligned copy of the
    # normalized feature DataFrame.
    #
    # This is the critical fix.
    # -----------------------------------------------------

    destination_aligned_features = (
        normalized_features
        .copy()
    )


    # -----------------------------------------------------
    # Verify that the number of rows matches the trusted
    # destination list.
    # -----------------------------------------------------

    if len(
        destination_aligned_features
    ) != len(destinations):

        raise ValueError(
            "Normalized feature row count does not "
            "match destination count."
        )


    # -----------------------------------------------------
    # Replace the RangeIndex with the actual destination
    # names.
    #
    # Row 0 -> Agra
    # Row 1 -> Ahmedabad
    # ...
    # Row 49 -> Wayanad
    # -----------------------------------------------------

    destination_aligned_features.index = (
        destinations
    )


    # -----------------------------------------------------
    # Create the weighted score vector.
    #
    # It now uses exactly the same destination index as
    # destination_aligned_features.
    # -----------------------------------------------------

    weighted_total = pd.Series(
        0.0,
        index=destinations,
        dtype=float
    )


    active_total = 0.0


    # -----------------------------------------------------
    # STEP 2 — Process every preference group.
    # -----------------------------------------------------

    for (
        group_name,
        feature_columns
    ) in preference_groups.items():

        # -------------------------------------------------
        # Get the user's importance for this group.
        #
        # If the group is not provided, treat it as zero.
        # -------------------------------------------------

        user_weight = float(
            user_preferences.get(
                group_name,
                0.0
            )
        )


        # -------------------------------------------------
        # Ignore inactive groups.
        # -------------------------------------------------

        if user_weight <= 0:

            continue


        # -------------------------------------------------
        # Keep only features that actually exist in the
        # saved normalized feature dataset.
        # -------------------------------------------------

        available_columns = [

            column

            for column in feature_columns

            if column
            in destination_aligned_features.columns
        ]


        # -------------------------------------------------
        # If none of the group's features are available,
        # skip the group safely.
        # -------------------------------------------------

        if not available_columns:

            continue


        # -------------------------------------------------
        # Extract the group's features.
        #
        # Because the DataFrame now has destination names
        # as its index, group_values will also have
        # destination names as its index.
        # -------------------------------------------------

        group_values = (
            destination_aligned_features[
                available_columns
            ]
            .copy()
        )


        # -------------------------------------------------
        # Apply preference direction.
        #
        # "higher":
        #     higher normalized value is better.
        #
        # "lower":
        #     lower normalized value is better, so invert.
        # -------------------------------------------------

        for column in available_columns:

            direction = (
                preference_directions.get(
                    column,
                    "higher"
                )
            )


            if direction == "lower":

                group_values[column] = (
                    1.0
                    -
                    group_values[column]
                )


        # -------------------------------------------------
        # Calculate the average group score.
        #
        # The result retains destination names as its index.
        # -------------------------------------------------

        group_score = (
            group_values
            .mean(
                axis=1,
                skipna=True
            )
        )


        # -------------------------------------------------
        # If all features in a row are missing, treat the
        # group score as zero rather than NaN.
        # -------------------------------------------------

        group_score = (
            group_score
            .fillna(0.0)
        )


        # -------------------------------------------------
        # Add weighted group contribution.
        #
        # Both Series now have:
        #
        #     Agra
        #     Ahmedabad
        #     ...
        #
        # so Pandas alignment is safe.
        # -------------------------------------------------

        weighted_total = (
            weighted_total
            +
            group_score
            *
            user_weight
        )


        # -------------------------------------------------
        # Keep track of the total active user weight.
        # -------------------------------------------------

        active_total += user_weight


    # -----------------------------------------------------
    # STEP 3 — ZERO-WEIGHT SAFETY
    #
    # If the user selected no preference importance at all,
    # return zero scores.
    #
    # DO NOT divide by zero.
    # -----------------------------------------------------

    if active_total <= 0:

        return pd.Series(
            0.0,
            index=destinations,
            dtype=float
        )


    # -----------------------------------------------------
    # STEP 4 — Normalize by active preference weight.
    # -----------------------------------------------------

    final_score = (
        weighted_total
        /
        active_total
    )


    # -----------------------------------------------------
    # STEP 5 — Final numerical safety.
    #
    # No NaN.
    # No infinity.
    # Score remains within [0, 1].
    # -----------------------------------------------------

    final_score = (
        final_score
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .fillna(0.0)
        .clip(
            lower=0.0,
            upper=1.0
        )
    )


    # -----------------------------------------------------
    # STEP 6 — Final index validation.
    # -----------------------------------------------------

    if list(
        final_score.index
    ) != destinations:

        raise ValueError(
            "Preference score destination alignment "
            "failed."
        )


    if len(
        final_score
    ) != len(destinations):

        raise ValueError(
            "Preference score count does not match "
            "destination count."
        )


    return final_score


print(
    "=" * 60
)

print(
    "✓ FIXED ZERO-SAFE PREFERENCE SCORING FUNCTION CREATED"
)

print(
    "=" * 60
)

✓ FIXED ZERO-SAFE PREFERENCE SCORING FUNCTION CREATED


In [19]:
# =========================================================
# CELL 11 — ZERO-SAFE INTEREST SCORING
# =========================================================
#
# This function calculates how well each destination matches
# the user's interests.
#
# It is explicitly protected against:
#
#     total interest weight = 0
#
# which was the source of the previous NaN problem.
# =========================================================


def calculate_interest_score(
    user_interests
):
    """
    Calculate personalized interest scores.

    Parameters
    ----------
    user_interests : dict
        Dictionary containing importance values for:
            nature
            sightseeing
            water_coastal
            wildlife

    Returns
    -------
    pandas.Series
        Finite interest score for every destination.
    """

    # -----------------------------------------------------
    # Start with zero scores.
    # -----------------------------------------------------

    weighted_total = pd.Series(
        0.0,
        index=destinations,
        dtype=float
    )


    active_total = 0.0


    # -----------------------------------------------------
    # Map interest names to the columns in the destination
    # profile DataFrame.
    # -----------------------------------------------------

    interest_column_map = {

        "nature":
            "nature_score",

        "sightseeing":
            "sightseeing_score",

        "water_coastal":
            "water_coastal_score",

        "wildlife":
            "wildlife_score"
    }


    # -----------------------------------------------------
    # Process every interest profile.
    # -----------------------------------------------------

    for interest_name in interest_profiles:

        user_weight = float(
            user_interests.get(
                interest_name,
                0.0
            )
        )


        # -------------------------------------------------
        # Ignore inactive interests.
        # -------------------------------------------------

        if user_weight <= 0:

            continue


        score_column = (
            interest_column_map.get(
                interest_name
            )
        )


        if score_column is None:

            continue


        if (
            score_column
            not in destination_profile_scores.columns
        ):

            continue


        # -------------------------------------------------
        # Retrieve destination scores.
        # -------------------------------------------------

        destination_scores = pd.Series(

            destination_profile_scores[
                score_column
            ].to_numpy(
                dtype=float
            ),

            index=destinations,

            dtype=float
        )


        # -------------------------------------------------
        # Missing interest profile values become zero.
        # -------------------------------------------------

        destination_scores = (
            destination_scores
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
            .fillna(0.0)
            .clip(
                0.0,
                1.0
            )
        )


        # -------------------------------------------------
        # Add weighted contribution.
        # -------------------------------------------------

        weighted_total = (
            weighted_total
            +
            destination_scores
            * user_weight
        )


        active_total += user_weight


    # -----------------------------------------------------
    # ZERO-WEIGHT SAFETY
    # -----------------------------------------------------

    if active_total <= 0:

        return pd.Series(
            0.0,
            index=destinations,
            dtype=float
        )


    # -----------------------------------------------------
    # Normalize active interest weights.
    # -----------------------------------------------------

    final_score = (
        weighted_total
        /
        active_total
    )


    # -----------------------------------------------------
    # Final numerical safety.
    # -----------------------------------------------------

    final_score = (
        final_score
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .fillna(0.0)
        .clip(
            0.0,
            1.0
        )
    )


    return final_score

In [20]:
# =========================================================
# CELL 12 — SIMILARITY SCORING
# =========================================================
#
# Calculates normalized similarity to a reference destination.
#
# Example:
#
#     reference_destination = "Goa"
#
# Goa itself will receive similarity = 1.0, but the final
# recommendation engine will remove Goa from the output.
# =========================================================


def calculate_similarity_score(
    reference_destination
):
    """
    Calculate normalized similarity scores.

    Parameters
    ----------
    reference_destination : str
        Destination used as the similarity reference.

    Returns
    -------
    pandas.Series
        Similarity score in [0, 1] for every destination.
    """

    # -----------------------------------------------------
    # Validate reference destination.
    # -----------------------------------------------------

    if reference_destination not in destinations:

        raise ValueError(
            f"Unknown reference destination: "
            f"{reference_destination}"
        )


    # -----------------------------------------------------
    # Extract similarity row using the trusted destination
    # order.
    # -----------------------------------------------------

    raw_similarity = (
        destination_similarity
        .loc[
            reference_destination,
            destinations
        ]
        .astype(float)
    )


    # -----------------------------------------------------
    # Normalize similarity into [0, 1].
    # -----------------------------------------------------

    minimum = float(
        raw_similarity.min()
    )

    maximum = float(
        raw_similarity.max()
    )


    if maximum == minimum:

        normalized = pd.Series(
            0.5,
            index=destinations,
            dtype=float
        )

    else:

        normalized = (

            raw_similarity
            -
            minimum

        ) / (

            maximum
            -
            minimum
        )


    # -----------------------------------------------------
    # Numerical safety.
    # -----------------------------------------------------

    normalized = (
        normalized
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .fillna(0.0)
        .clip(
            0.0,
            1.0
        )
    )


    return normalized

In [21]:
# =========================================================
# CELL 13 — CORE SCORING FUNCTION TEST
# =========================================================
#
# We test:
#
#   1. All-zero preferences
#   2. Budget-only preferences
#   3. Nature-only interests
#   4. All-maximum profile
#   5. Similarity to Goa
#
# The most important check is that there are NO NaNs.
# =========================================================


print("=" * 60)
print("CORE SCORING FUNCTION TEST")
print("=" * 60)


# ---------------------------------------------------------
# Test 1 — All-zero preferences.
# ---------------------------------------------------------

zero_preferences = {

    "budget": 0.0,

    "flight": 0.0,

    "accommodation": 0.0,

    "weather": 0.0,

    "destination_characteristics": 0.0
}


zero_preference_scores = (
    calculate_preference_score(
        zero_preferences
    )
)


print(
    "\nAll-zero preference profile:"
)

print(
    "NaN:",
    zero_preference_scores.isna().sum()
)

print(
    "Infinity:",
    np.isinf(
        zero_preference_scores
    ).sum()
)

print(
    "Unique scores:",
    zero_preference_scores.nunique()
)


# ---------------------------------------------------------
# Test 2 — Budget-only profile.
# ---------------------------------------------------------

budget_only = {

    "budget": 1.0,

    "flight": 0.0,

    "accommodation": 0.0,

    "weather": 0.0,

    "destination_characteristics": 0.0
}


budget_scores = (
    calculate_preference_score(
        budget_only
    )
)


print(
    "\nBudget-only profile:"
)

print(
    "NaN:",
    budget_scores.isna().sum()
)

print(
    "Maximum:",
    budget_scores.max()
)


# ---------------------------------------------------------
# Test 3 — Nature-only profile.
# ---------------------------------------------------------

nature_only = {

    "nature": 1.0,

    "sightseeing": 0.0,

    "water_coastal": 0.0,

    "wildlife": 0.0
}


nature_scores = (
    calculate_interest_score(
        nature_only
    )
)


print(
    "\nNature-only profile:"
)

print(
    "NaN:",
    nature_scores.isna().sum()
)

print(
    "Maximum:",
    nature_scores.max()
)


# ---------------------------------------------------------
# Test 4 — All maximum.
# ---------------------------------------------------------

all_preferences = {

    group: 1.0

    for group in preference_groups
}


all_interests = {

    interest: 1.0

    for interest in interest_profiles
}


all_preference_scores = (
    calculate_preference_score(
        all_preferences
    )
)


all_interest_scores = (
    calculate_interest_score(
        all_interests
    )
)


print(
    "\nAll-maximum profile:"
)

print(
    "Preference NaN:",
    all_preference_scores.isna().sum()
)

print(
    "Interest NaN:",
    all_interest_scores.isna().sum()
)


# ---------------------------------------------------------
# Test 5 — Similarity to Goa.
# ---------------------------------------------------------

goa_similarity = (
    calculate_similarity_score(
        "Goa"
    )
)


print(
    "\nSimilarity to Goa:"
)

print(
    "NaN:",
    goa_similarity.isna().sum()
)

print(
    "Maximum:",
    goa_similarity.max()
)

print(
    "Goa similarity:",
    goa_similarity.loc["Goa"]
)


# ---------------------------------------------------------
# Final validation.
# ---------------------------------------------------------

all_test_series = [

    zero_preference_scores,

    budget_scores,

    nature_scores,

    all_preference_scores,

    all_interest_scores,

    goa_similarity
]


for series in all_test_series:

    if series.isna().any():

        raise ValueError(
            "NaN detected in core scoring function."
        )


    if np.isinf(
        series.to_numpy()
    ).any():

        raise ValueError(
            "Infinity detected in core scoring function."
        )


print(
    "\n✓ ALL CORE SCORING FUNCTIONS PASSED"
)

CORE SCORING FUNCTION TEST

All-zero preference profile:
NaN: 0
Infinity: 0
Unique scores: 1

Budget-only profile:
NaN: 0
Maximum: 0.0

Nature-only profile:
NaN: 0
Maximum: 0.6194589175876536

All-maximum profile:
Preference NaN: 0
Interest NaN: 0

Similarity to Goa:
NaN: 0
Maximum: 1.0
Goa similarity: 1.0

✓ ALL CORE SCORING FUNCTIONS PASSED


In [26]:
# =========================================================
# CELL 10A — VERIFY PREFERENCE INDEX ALIGNMENT
# =========================================================


print("=" * 60)
print("VERIFYING PREFERENCE SCORE ALIGNMENT")
print("=" * 60)


# ---------------------------------------------------------
# Test zero profile.
# ---------------------------------------------------------

zero_preferences = {

    "budget": 0.0,

    "flight": 0.0,

    "accommodation": 0.0,

    "weather": 0.0,

    "destination_characteristics": 0.0
}


zero_scores = (
    calculate_preference_score(
        zero_preferences
    )
)


print(
    "\nZERO PROFILE"
)

print(
    "Length:",
    len(zero_scores)
)

print(
    "Index type:",
    type(zero_scores.index)
)

print(
    "NaN:",
    zero_scores.isna().sum()
)

print(
    "Infinity:",
    np.isinf(
        zero_scores.to_numpy()
    ).sum()
)

print(
    "Unique scores:",
    zero_scores.nunique()
)


# ---------------------------------------------------------
# Test budget-only profile.
# ---------------------------------------------------------

budget_only = {

    "budget": 1.0,

    "flight": 0.0,

    "accommodation": 0.0,

    "weather": 0.0,

    "destination_characteristics": 0.0
}


budget_scores = (
    calculate_preference_score(
        budget_only
    )
)


print(
    "\nBUDGET-ONLY PROFILE"
)

print(
    "Length:",
    len(budget_scores)
)

print(
    "NaN:",
    budget_scores.isna().sum()
)

print(
    "Infinity:",
    np.isinf(
        budget_scores.to_numpy()
    ).sum()
)

print(
    "Minimum:",
    budget_scores.min()
)

print(
    "Maximum:",
    budget_scores.max()
)


print(
    "\nTop 10 budget destinations:"
)

print(
    budget_scores
    .sort_values(
        ascending=False
    )
    .head(10)
)


# ---------------------------------------------------------
# Verify destination alignment.
# ---------------------------------------------------------

if list(
    budget_scores.index
) != destinations:

    raise ValueError(
        "Budget score index does not match "
        "trusted destination order."
    )


# ---------------------------------------------------------
# Verify finite scores.
# ---------------------------------------------------------

if budget_scores.isna().any():

    raise ValueError(
        "Budget-only profile produced NaN."
    )


if np.isinf(
    budget_scores.to_numpy()
).any():

    raise ValueError(
        "Budget-only profile produced infinity."
    )


# ---------------------------------------------------------
# Final validation.
# ---------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "✓ PREFERENCE ALIGNMENT FIX VERIFIED"
)

print(
    "=" * 60
)

VERIFYING PREFERENCE SCORE ALIGNMENT

ZERO PROFILE
Length: 50
Index type: <class 'pandas.Index'>
NaN: 0
Infinity: 0
Unique scores: 1

BUDGET-ONLY PROFILE
Length: 50
NaN: 0
Infinity: 0
Minimum: 0.0
Maximum: 0.6305528929512362

Top 10 budget destinations:
Rishikesh      0.630553
Ooty           0.523525
Dharamshala    0.417008
Srinagar       0.345115
Ranthambore    0.334979
Udaipur        0.263770
Munnar         0.224020
Jaipur         0.168911
Alappuzha      0.146660
Delhi          0.146505
dtype: float64

✓ PREFERENCE ALIGNMENT FIX VERIFIED


In [27]:
# =========================================================
# PHASE 9B — PRODUCTION RECOMMENDATION ENGINE
# CELL 14 — GET RECOMMENDATIONS
# =========================================================
#
# This is the main recommendation function for the new
# production notebook.
#
# IMPORTANT:
#
# It uses ONLY:
#
#   - saved artifacts
#   - configuration
#   - the three validated scoring functions
#
# It does NOT depend on variables from Notebook 08.
# =========================================================


def get_recommendations(
    user_preferences,
    user_interests,
    reference_destination=None,
    mode="hybrid",
    top_k=10
):
    """
    Generate personalized travel recommendations.

    Parameters
    ----------
    user_preferences : dict
        Importance weights for travel preference groups.

    user_interests : dict
        Importance weights for destination interests.

    reference_destination : str or None
        Destination used for similarity-based recommendations.

    mode : str
        One of:
            "preference"
            "similarity"
            "hybrid"

    top_k : int
        Number of recommendations to return.

    Returns
    -------
    pandas.DataFrame
        Ranked destination recommendations.
    """

    # -----------------------------------------------------
    # Validate recommendation mode.
    # -----------------------------------------------------

    valid_modes = [
        "preference",
        "similarity",
        "hybrid"
    ]


    if mode not in valid_modes:

        raise ValueError(
            f"Invalid mode '{mode}'. "
            f"Choose from {valid_modes}."
        )


    # -----------------------------------------------------
    # Validate top_k.
    # -----------------------------------------------------

    if not isinstance(
        top_k,
        int
    ) or top_k <= 0:

        raise ValueError(
            "top_k must be a positive integer."
        )


    # -----------------------------------------------------
    # Validate reference destination.
    #
    # Similarity and hybrid modes require a reference.
    # Preference mode does not.
    # -----------------------------------------------------

    if mode in [
        "similarity",
        "hybrid"
    ]:

        if reference_destination is None:

            raise ValueError(
                "reference_destination is required "
                f"for {mode} mode."
            )


        if (
            reference_destination
            not in destinations
        ):

            raise ValueError(
                f"Unknown reference destination: "
                f"{reference_destination}"
            )


    # -----------------------------------------------------
    # Calculate personalized travel preference score.
    # -----------------------------------------------------

    preference_score = (
        calculate_preference_score(
            user_preferences
        )
    )


    # -----------------------------------------------------
    # Calculate personalized interest score.
    # -----------------------------------------------------

    interest_score = (
        calculate_interest_score(
            user_interests
        )
    )


    # -----------------------------------------------------
    # Validate scoring outputs.
    # -----------------------------------------------------

    for score_name, scores in [

        (
            "preference",
            preference_score
        ),

        (
            "interest",
            interest_score
        )

    ]:

        if len(scores) != len(destinations):

            raise ValueError(
                f"{score_name} score count does not "
                "match destination count."
            )


        if scores.isna().any():

            raise ValueError(
                f"NaN detected in {score_name} scores."
            )


        if np.isinf(
            scores.to_numpy()
        ).any():

            raise ValueError(
                f"Infinity detected in "
                f"{score_name} scores."
            )


    # -----------------------------------------------------
    # Combine travel preferences and interests.
    #
    # Travel preferences = 60%
    # Interests          = 40%
    # -----------------------------------------------------

    personalized_score = (

        preference_score
        *
        travel_factor_weight

        +

        interest_score
        *
        interest_factor_weight
    )


    # -----------------------------------------------------
    # Initialize similarity scores to zero.
    #
    # This ensures preference-only mode does not require
    # a similarity calculation.
    # -----------------------------------------------------

    similarity_score = pd.Series(
        0.0,
        index=destinations,
        dtype=float
    )


    # -----------------------------------------------------
    # Calculate similarity when requested.
    # -----------------------------------------------------

    if mode in [
        "similarity",
        "hybrid"
    ]:

        similarity_score = (
            calculate_similarity_score(
                reference_destination
            )
        )


    # -----------------------------------------------------
    # Calculate final score according to mode.
    # -----------------------------------------------------

    if mode == "preference":

        final_score = (
            personalized_score
        )


    elif mode == "similarity":

        final_score = (
            similarity_score
        )


    else:

        # -------------------------------------------------
        # Hybrid recommendation.
        #
        # Personalized score = 70%
        # Similarity          = 30%
        # -------------------------------------------------

        final_score = (

            personalized_score
            *
            hybrid_preference_weight

            +

            similarity_score
            *
            hybrid_similarity_weight
        )


    # -----------------------------------------------------
    # Numerical safety.
    # -----------------------------------------------------

    score_columns = {

        "personalized_preference_score":
            preference_score,

        "personalized_interest_score":
            interest_score,

        "similarity_score":
            similarity_score,

        "final_recommendation_score":
            final_score
    }


    for score_name, scores in (
        score_columns.items()
    ):

        if scores.isna().any():

            raise ValueError(
                f"NaN detected in final "
                f"{score_name}."
            )


        if np.isinf(
            scores.to_numpy()
        ).any():

            raise ValueError(
                f"Infinity detected in final "
                f"{score_name}."
            )


    # -----------------------------------------------------
    # Construct result DataFrame.
    #
    # IMPORTANT:
    # We explicitly use destination names as the source
    # for every row. This prevents the RangeIndex/NaN
    # alignment problem from Notebook 08.
    # -----------------------------------------------------

    result = pd.DataFrame({

        "destination":
            destinations,

        "personalized_preference_score":
            preference_score.to_numpy(),

        "personalized_interest_score":
            interest_score.to_numpy(),

        "similarity_score":
            similarity_score.to_numpy(),

        "final_recommendation_score":
            final_score.to_numpy()
    })


    # -----------------------------------------------------
    # Remove reference destination.
    #
    # A destination should never recommend itself.
    # -----------------------------------------------------

    if reference_destination is not None:

        result = result[
            result[
                "destination"
            ]
            != reference_destination
        ].copy()


    # -----------------------------------------------------
    # Sort by final score.
    # -----------------------------------------------------

    result = (
        result
        .sort_values(
            "final_recommendation_score",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


    # -----------------------------------------------------
    # Add ranking.
    # -----------------------------------------------------

    result.insert(
        0,
        "rank",
        range(
            1,
            len(result) + 1
        )
    )


    # -----------------------------------------------------
    # Return Top-K.
    # -----------------------------------------------------

    return result.head(
        min(
            top_k,
            len(result)
        )
    )

In [28]:
# =========================================================
# CELL 15 — CREATE TEST USER PROFILE
# =========================================================
#
# This represents a traveler who:
#
#   - strongly cares about budget
#   - moderately cares about weather
#   - moderately cares about destination characteristics
#   - has strong nature interest
#   - moderate wildlife interest
#   - low coastal interest
# =========================================================


test_user_preferences = {

    "budget": 0.9,

    "flight": 0.6,

    "accommodation": 0.6,

    "weather": 0.8,

    "destination_characteristics": 0.8
}


test_user_interests = {

    "nature": 1.0,

    "sightseeing": 0.5,

    "water_coastal": 0.2,

    "wildlife": 0.6
}


print("=" * 60)
print("TEST USER PROFILE")
print("=" * 60)


print(
    "\nTravel preferences:"
)

for key, value in (
    test_user_preferences.items()
):

    print(
        f"  {key}: {value}"
    )


print(
    "\nInterests:"
)

for key, value in (
    test_user_interests.items()
):

    print(
        f"  {key}: {value}"
    )

TEST USER PROFILE

Travel preferences:
  budget: 0.9
  flight: 0.6
  accommodation: 0.6
  weather: 0.8
  destination_characteristics: 0.8

Interests:
  nature: 1.0
  sightseeing: 0.5
  water_coastal: 0.2
  wildlife: 0.6


In [29]:
# =========================================================
# CELL 16 — PREFERENCE-ONLY RECOMMENDATION TEST
# =========================================================


preference_recommendations = (
    get_recommendations(

        user_preferences=
            test_user_preferences,

        user_interests=
            test_user_interests,

        mode="preference",

        top_k=10
    )
)


print("=" * 60)
print("PREFERENCE-ONLY RECOMMENDATIONS")
print("=" * 60)


print(
    preference_recommendations[
        [
            "rank",
            "destination",
            "personalized_preference_score",
            "personalized_interest_score",
            "final_recommendation_score"
        ]
    ].to_string(
        index=False
    )
)

PREFERENCE-ONLY RECOMMENDATIONS
 rank destination  personalized_preference_score  personalized_interest_score  final_recommendation_score
    1      Mumbai                       0.334219                     0.522352                    0.409472
    2       Delhi                       0.384493                     0.408991                    0.394292
    3    Srinagar                       0.387136                     0.349341                    0.372018
    4      Jaipur                       0.316367                     0.410318                    0.353947
    5   Rishikesh                       0.384775                     0.303697                    0.352343
    6     Udaipur                       0.386985                     0.297781                    0.351303
    7      Manali                       0.240988                     0.487155                    0.339455
    8   Bengaluru                       0.283334                     0.401584                    0.330634
    9       Ko

In [30]:
# =========================================================
# CELL 17 — HYBRID RECOMMENDATION TEST
# =========================================================
#
# Reference destination = Goa.
#
# Goa should NOT appear in the final results.
# =========================================================


hybrid_recommendations = (
    get_recommendations(

        user_preferences=
            test_user_preferences,

        user_interests=
            test_user_interests,

        reference_destination=
            "Goa",

        mode="hybrid",

        top_k=10
    )
)


print("=" * 60)
print("HYBRID RECOMMENDATIONS — SIMILAR TO GOA")
print("=" * 60)


print(
    hybrid_recommendations[
        [
            "rank",
            "destination",
            "personalized_preference_score",
            "personalized_interest_score",
            "similarity_score",
            "final_recommendation_score"
        ]
    ].to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Verify that Goa was excluded.
# ---------------------------------------------------------

if (
    "Goa"
    in hybrid_recommendations[
        "destination"
    ].values
):

    raise ValueError(
        "Reference destination Goa "
        "was not excluded."
    )


print(
    "\n✓ Goa correctly excluded."
)

HYBRID RECOMMENDATIONS — SIMILAR TO GOA
 rank destination  personalized_preference_score  personalized_interest_score  similarity_score  final_recommendation_score
    1      Mumbai                       0.334219                     0.522352          0.730195                    0.505689
    2       Delhi                       0.384493                     0.408991          0.478832                    0.419654
    3      Jaipur                       0.316367                     0.410318          0.460086                    0.385789
    4    Srinagar                       0.387136                     0.349341          0.396834                    0.379463
    5   Bengaluru                       0.283334                     0.401584          0.483953                    0.376630
    6     Udaipur                       0.386985                     0.297781          0.372920                    0.357789
    7   Alappuzha                       0.252643                     0.261628          0.579

In [31]:
# =========================================================
# CELL 18 — PRODUCTION ENGINE SANITY TEST
# =========================================================
#
# We validate:
#
#   ✓ Correct number of destinations
#   ✓ No NaN scores
#   ✓ No infinite scores
#   ✓ Scores are within valid ranges
#   ✓ Ranking is descending
#   ✓ Reference destination is excluded
# =========================================================


print("=" * 60)
print("PRODUCTION ENGINE SANITY TEST")
print("=" * 60)


# ---------------------------------------------------------
# Test expected number of rows.
# ---------------------------------------------------------

if len(
    hybrid_recommendations
) != 10:

    raise ValueError(
        "Expected 10 recommendations."
    )


# ---------------------------------------------------------
# Validate score columns.
# ---------------------------------------------------------

score_columns = [

    "personalized_preference_score",

    "personalized_interest_score",

    "similarity_score",

    "final_recommendation_score"
]


for column in score_columns:

    nan_count = (
        hybrid_recommendations[
            column
        ]
        .isna()
        .sum()
    )


    infinite_count = np.isinf(
        hybrid_recommendations[
            column
        ]
        .to_numpy()
    ).sum()


    if nan_count > 0:

        raise ValueError(
            f"{column} contains NaN."
        )


    if infinite_count > 0:

        raise ValueError(
            f"{column} contains infinity."
        )


    if (
        (
            hybrid_recommendations[
                column
            ] < 0
        )
        |
        (
            hybrid_recommendations[
                column
            ] > 1
        )
    ).any():

        raise ValueError(
            f"{column} contains values "
            "outside [0, 1]."
        )


# ---------------------------------------------------------
# Verify descending ranking.
# ---------------------------------------------------------

scores = (
    hybrid_recommendations[
        "final_recommendation_score"
    ]
    .to_numpy()
)


if not np.all(
    scores[:-1] >= scores[1:]
):

    raise ValueError(
        "Final recommendations are not "
        "sorted in descending order."
    )


# ---------------------------------------------------------
# Verify unique destinations.
# ---------------------------------------------------------

if (
    hybrid_recommendations[
        "destination"
    ]
    .nunique()
    !=
    len(hybrid_recommendations)
):

    raise ValueError(
        "Duplicate destinations detected."
    )


# ---------------------------------------------------------
# Final success.
# ---------------------------------------------------------

print(
    "\n✓ 10 recommendations returned"
)

print(
    "✓ No NaN scores"
)

print(
    "✓ No infinite scores"
)

print(
    "✓ All scores within [0, 1]"
)

print(
    "✓ Ranking is correctly ordered"
)

print(
    "✓ Destinations are unique"
)

print(
    "✓ Reference destination excluded"
)


print(
    "\n" + "=" * 60
)

print(
    "✓ PRODUCTION RECOMMENDATION ENGINE PASSED"
)

print(
    "=" * 60
)

PRODUCTION ENGINE SANITY TEST

✓ 10 recommendations returned
✓ No NaN scores
✓ No infinite scores
✓ All scores within [0, 1]
✓ Ranking is correctly ordered
✓ Destinations are unique
✓ Reference destination excluded

✓ PRODUCTION RECOMMENDATION ENGINE PASSED


In [32]:
# =========================================================
# PHASE 9C — USER INPUT LAYER
# CELL 19 — HUMAN-READABLE IMPORTANCE MAPPING
# =========================================================
#
# Users should not need to know that the model expects
# numerical values such as 0.8 or 1.0.
#
# Instead, they can choose:
#
#     None
#     Low
#     Medium
#     High
#     Very High
#
# This cell converts those human-readable choices into the
# numerical weights expected by the recommendation engine.
# =========================================================


IMPORTANCE_WEIGHTS = {

    "none": 0.0,

    "low": 0.2,

    "medium": 0.6,

    "high": 0.8,

    "very high": 1.0
}


# ---------------------------------------------------------
# Also support common capitalization variations.
#
# Internally we always convert the user's input to lowercase.
# ---------------------------------------------------------

print("=" * 60)
print("USER IMPORTANCE SCALE")
print("=" * 60)


for label, weight in IMPORTANCE_WEIGHTS.items():

    print(
        f"{label.title():<12} → {weight}"
    )


print(
    "\n✓ HUMAN-READABLE IMPORTANCE MAPPING READY"
)

USER IMPORTANCE SCALE
None         → 0.0
Low          → 0.2
Medium       → 0.6
High         → 0.8
Very High    → 1.0

✓ HUMAN-READABLE IMPORTANCE MAPPING READY


In [33]:
# =========================================================
# CELL 20 — CONVERT HUMAN INPUT TO MODEL WEIGHT
# =========================================================
#
# This function:
#
#     "High"      → 0.8
#     "Medium"    → 0.6
#     "Very High" → 1.0
#
# It also validates invalid inputs instead of silently
# accepting them.
# =========================================================


def importance_to_weight(
    importance
):
    """
    Convert a human-readable importance level to a
    numerical model weight.

    Accepted values:
        None
        Low
        Medium
        High
        Very High

    Returns
    -------
    float
        Numerical weight in [0, 1].
    """

    # -----------------------------------------------------
    # Make sure the value is a string.
    # -----------------------------------------------------

    if not isinstance(
        importance,
        str
    ):

        raise ValueError(
            "Importance must be provided as text."
        )


    # -----------------------------------------------------
    # Normalize user input.
    #
    # Example:
    #
    #     "  Very High "
    #
    # becomes:
    #
    #     "very high"
    # -----------------------------------------------------

    normalized_input = (
        importance
        .strip()
        .lower()
    )


    # -----------------------------------------------------
    # Validate the input.
    # -----------------------------------------------------

    if (
        normalized_input
        not in IMPORTANCE_WEIGHTS
    ):

        valid_choices = ", ".join(
            label.title()
            for label
            in IMPORTANCE_WEIGHTS
        )


        raise ValueError(
            "Invalid importance level: "
            f"'{importance}'. "
            f"Choose from: {valid_choices}."
        )


    # -----------------------------------------------------
    # Return corresponding model weight.
    # -----------------------------------------------------

    return IMPORTANCE_WEIGHTS[
        normalized_input
    ]


# ---------------------------------------------------------
# Test the conversion function.
# ---------------------------------------------------------

print("=" * 60)
print("IMPORTANCE CONVERSION TEST")
print("=" * 60)


test_levels = [

    "None",
    "Low",
    "Medium",
    "High",
    "Very High"
]


for level in test_levels:

    print(
        f"{level:<12} → "
        f"{importance_to_weight(level)}"
    )


print(
    "\n✓ IMPORTANCE CONVERSION FUNCTION READY"
)

IMPORTANCE CONVERSION TEST
None         → 0.0
Low          → 0.2
Medium       → 0.6
High         → 0.8
Very High    → 1.0

✓ IMPORTANCE CONVERSION FUNCTION READY


In [34]:
# =========================================================
# CELL 21 — USER PROFILE VALIDATION
# =========================================================
#
# This function validates the structure of a user's profile
# before it reaches the recommendation engine.
#
# This creates a clean boundary:
#
#     USER INPUT
#          ↓
#     VALIDATION
#          ↓
#     RECOMMENDATION ENGINE
# =========================================================


REQUIRED_PREFERENCE_GROUPS = [

    "budget",

    "flight",

    "accommodation",

    "weather",

    "destination_characteristics"
]


REQUIRED_INTEREST_GROUPS = [

    "nature",

    "sightseeing",

    "water_coastal",

    "wildlife"
]


def validate_user_profile(
    user_preferences,
    user_interests
):
    """
    Validate a numerical user profile.

    Parameters
    ----------
    user_preferences : dict
        Numerical weights for travel preferences.

    user_interests : dict
        Numerical weights for interests.

    Returns
    -------
    bool
        True when the profile is valid.
    """

    # -----------------------------------------------------
    # Validate preference object.
    # -----------------------------------------------------

    if not isinstance(
        user_preferences,
        dict
    ):

        raise ValueError(
            "user_preferences must be a dictionary."
        )


    # -----------------------------------------------------
    # Validate interest object.
    # -----------------------------------------------------

    if not isinstance(
        user_interests,
        dict
    ):

        raise ValueError(
            "user_interests must be a dictionary."
        )


    # -----------------------------------------------------
    # Check required preference groups.
    # -----------------------------------------------------

    missing_preferences = [

        group

        for group
        in REQUIRED_PREFERENCE_GROUPS

        if group
        not in user_preferences
    ]


    if missing_preferences:

        raise ValueError(
            "Missing preference groups: "
            f"{missing_preferences}"
        )


    # -----------------------------------------------------
    # Check required interest groups.
    # -----------------------------------------------------

    missing_interests = [

        interest

        for interest
        in REQUIRED_INTEREST_GROUPS

        if interest
        not in user_interests
    ]


    if missing_interests:

        raise ValueError(
            "Missing interest groups: "
            f"{missing_interests}"
        )


    # -----------------------------------------------------
    # Validate every preference weight.
    # -----------------------------------------------------

    for group, weight in (
        user_preferences.items()
    ):

        if not isinstance(
            weight,
            (int, float)
        ):

            raise ValueError(
                f"Preference '{group}' "
                "must be numeric."
            )


        if not np.isfinite(
            weight
        ):

            raise ValueError(
                f"Preference '{group}' "
                "must be finite."
            )


        if not (
            0.0
            <=
            float(weight)
            <=
            1.0
        ):

            raise ValueError(
                f"Preference '{group}' "
                "must be between 0 and 1."
            )


    # -----------------------------------------------------
    # Validate every interest weight.
    # -----------------------------------------------------

    for interest, weight in (
        user_interests.items()
    ):

        if not isinstance(
            weight,
            (int, float)
        ):

            raise ValueError(
                f"Interest '{interest}' "
                "must be numeric."
            )


        if not np.isfinite(
            weight
        ):

            raise ValueError(
                f"Interest '{interest}' "
                "must be finite."
            )


        if not (
            0.0
            <=
            float(weight)
            <=
            1.0
        ):

            raise ValueError(
                f"Interest '{interest}' "
                "must be between 0 and 1."
            )


    return True


print(
    "=" * 60
)

print(
    "✓ USER PROFILE VALIDATION FUNCTION CREATED"
)

print(
    "=" * 60
)

✓ USER PROFILE VALIDATION FUNCTION CREATED


In [35]:
# =========================================================
# CELL 22 — BUILD HUMAN-FRIENDLY USER PROFILE
# =========================================================
#
# This function accepts values such as:
#
#     "Very High"
#     "High"
#     "Medium"
#     "Low"
#     "None"
#
# and converts them into the numerical representation used
# by the recommendation engine.
# =========================================================


def build_user_profile(
    budget="Medium",
    flight="Medium",
    accommodation="Medium",
    weather="Medium",
    destination_characteristics="Medium",
    nature="Medium",
    sightseeing="Medium",
    water_coastal="Medium",
    wildlife="Medium"
):
    """
    Build a complete numerical user profile from
    human-readable importance levels.
    """

    # -----------------------------------------------------
    # Convert travel preferences.
    # -----------------------------------------------------

    user_preferences = {

        "budget":
            importance_to_weight(
                budget
            ),

        "flight":
            importance_to_weight(
                flight
            ),

        "accommodation":
            importance_to_weight(
                accommodation
            ),

        "weather":
            importance_to_weight(
                weather
            ),

        "destination_characteristics":
            importance_to_weight(
                destination_characteristics
            )
    }


    # -----------------------------------------------------
    # Convert interests.
    # -----------------------------------------------------

    user_interests = {

        "nature":
            importance_to_weight(
                nature
            ),

        "sightseeing":
            importance_to_weight(
                sightseeing
            ),

        "water_coastal":
            importance_to_weight(
                water_coastal
            ),

        "wildlife":
            importance_to_weight(
                wildlife
            )
    }


    # -----------------------------------------------------
    # Validate the generated numerical profile.
    # -----------------------------------------------------

    validate_user_profile(
        user_preferences,
        user_interests
    )


    return (
        user_preferences,
        user_interests
    )


print(
    "=" * 60
)

print(
    "✓ HUMAN-FRIENDLY PROFILE BUILDER CREATED"
)

print(
    "=" * 60
)

✓ HUMAN-FRIENDLY PROFILE BUILDER CREATED


In [36]:
# =========================================================
# CELL 23 — REALISTIC TRAVELER PROFILE TEST
# =========================================================


user_preferences, user_interests = (
    build_user_profile(

        # -------------------------------------------------
        # Travel preferences
        # -------------------------------------------------

        budget="Very High",

        flight="Medium",

        accommodation="Medium",

        weather="High",

        destination_characteristics="High",


        # -------------------------------------------------
        # Interests
        # -------------------------------------------------

        nature="Very High",

        sightseeing="Medium",

        water_coastal="Low",

        wildlife="Medium"
    )
)


print("=" * 60)
print("GENERATED USER PROFILE")
print("=" * 60)


print(
    "\nTravel preferences:"
)

for group, weight in (
    user_preferences.items()
):

    print(
        f"  {group:<32} {weight}"
    )


print(
    "\nInterests:"
)

for interest, weight in (
    user_interests.items()
):

    print(
        f"  {interest:<32} {weight}"
    )


print(
    "\n✓ USER PROFILE CREATED SUCCESSFULLY"
)

GENERATED USER PROFILE

Travel preferences:
  budget                           1.0
  flight                           0.6
  accommodation                    0.6
  weather                          0.8
  destination_characteristics      0.8

Interests:
  nature                           1.0
  sightseeing                      0.6
  water_coastal                    0.2
  wildlife                         0.6

✓ USER PROFILE CREATED SUCCESSFULLY


In [37]:
# =========================================================
# CELL 24 — USER PROFILE → RECOMMENDATION ENGINE
# =========================================================
#
# This is the first complete user-to-recommendation flow.
# =========================================================


recommendations = (
    get_recommendations(

        user_preferences=
            user_preferences,

        user_interests=
            user_interests,

        mode="preference",

        top_k=5
    )
)


print("=" * 60)
print("PERSONALIZED TRAVEL RECOMMENDATIONS")
print("=" * 60)


print(
    recommendations[
        [
            "rank",

            "destination",

            "personalized_preference_score",

            "personalized_interest_score",

            "final_recommendation_score"
        ]
    ]
    .to_string(
        index=False
    )
)


print(
    "\n✓ USER PROFILE SUCCESSFULLY CONNECTED "
    "TO RECOMMENDATION ENGINE"
)

PERSONALIZED TRAVEL RECOMMENDATIONS
 rank destination  personalized_preference_score  personalized_interest_score  final_recommendation_score
    1      Mumbai                       0.327480                     0.535142                    0.410545
    2       Delhi                       0.378230                     0.424713                    0.396823
    3    Srinagar                       0.386030                     0.358867                    0.375165
    4   Rishikesh                       0.391243                     0.308254                    0.358047
    5      Jaipur                       0.312487                     0.418880                    0.355044

✓ USER PROFILE SUCCESSFULLY CONNECTED TO RECOMMENDATION ENGINE


In [38]:
# =========================================================
# CELL 25 — HUMAN PROFILE + REFERENCE DESTINATION
# =========================================================
#
# Scenario:
#
#     "I want a destination similar to Goa, but based on
#      my own travel preferences and interests."
#
# This combines:
#
#     Personalization
#           +
#     Destination similarity
# =========================================================


hybrid_user_recommendations = (
    get_recommendations(

        user_preferences=
            user_preferences,

        user_interests=
            user_interests,

        reference_destination=
            "Goa",

        mode="hybrid",

        top_k=5
    )
)


print("=" * 60)
print("PERSONALIZED ALTERNATIVES TO GOA")
print("=" * 60)


print(
    hybrid_user_recommendations[
        [
            "rank",

            "destination",

            "personalized_preference_score",

            "personalized_interest_score",

            "similarity_score",

            "final_recommendation_score"
        ]
    ]
    .to_string(
        index=False
    )
)


# ---------------------------------------------------------
# Verify Goa was excluded.
# ---------------------------------------------------------

if (
    "Goa"
    in hybrid_user_recommendations[
        "destination"
    ].values
):

    raise ValueError(
        "Goa incorrectly appeared in alternatives."
    )


print(
    "\n✓ Goa correctly excluded"
)

print(
    "✓ Personalization + similarity working together"
)

PERSONALIZED ALTERNATIVES TO GOA
 rank destination  personalized_preference_score  personalized_interest_score  similarity_score  final_recommendation_score
    1      Mumbai                       0.327480                     0.535142          0.730195                    0.506440
    2       Delhi                       0.378230                     0.424713          0.478832                    0.421426
    3      Jaipur                       0.312487                     0.418880          0.460086                    0.386557
    4    Srinagar                       0.386030                     0.358867          0.396834                    0.381665
    5   Bengaluru                       0.276341                     0.419018          0.483953                    0.378574

✓ Goa correctly excluded
✓ Personalization + similarity working together


In [39]:
# =========================================================
# CELL 26 — USER INPUT PIPELINE VALIDATION
# =========================================================
#
# This validates the entire path:
#
# Human-readable input
#       ↓
# Numerical profile
#       ↓
# Recommendation engine
#       ↓
# Valid recommendations
# =========================================================


print("=" * 60)
print("USER INPUT PIPELINE VALIDATION")
print("=" * 60)


# ---------------------------------------------------------
# Verify preference weights.
# ---------------------------------------------------------

for group, weight in (
    user_preferences.items()
):

    if not (
        0.0
        <=
        weight
        <=
        1.0
    ):

        raise ValueError(
            f"Invalid preference weight: "
            f"{group} = {weight}"
        )


# ---------------------------------------------------------
# Verify interest weights.
# ---------------------------------------------------------

for interest, weight in (
    user_interests.items()
):

    if not (
        0.0
        <=
        weight
        <=
        1.0
    ):

        raise ValueError(
            f"Invalid interest weight: "
            f"{interest} = {weight}"
        )


# ---------------------------------------------------------
# Verify recommendations.
# ---------------------------------------------------------

if len(
    hybrid_user_recommendations
) != 5:

    raise ValueError(
        "Expected 5 recommendations."
    )


# ---------------------------------------------------------
# Check recommendation scores.
# ---------------------------------------------------------

for column in [

    "personalized_preference_score",

    "personalized_interest_score",

    "similarity_score",

    "final_recommendation_score"

]:

    if (
        hybrid_user_recommendations[
            column
        ]
        .isna()
        .any()
    ):

        raise ValueError(
            f"NaN found in {column}."
        )


    if np.isinf(
        hybrid_user_recommendations[
            column
        ].to_numpy()
    ).any():

        raise ValueError(
            f"Infinity found in {column}."
        )


# ---------------------------------------------------------
# Verify reference exclusion.
# ---------------------------------------------------------

if (
    "Goa"
    in hybrid_user_recommendations[
        "destination"
    ].values
):

    raise ValueError(
        "Reference destination was not excluded."
    )


print(
    "\n✓ Human input conversion passed"
)

print(
    "✓ Profile validation passed"
)

print(
    "✓ Recommendation generation passed"
)

print(
    "✓ Score validation passed"
)

print(
    "✓ Reference exclusion passed"
)


print(
    "\n" + "=" * 60
)

print(
    "✓ COMPLETE USER INPUT PIPELINE PASSED"
)

print(
    "=" * 60
)

USER INPUT PIPELINE VALIDATION

✓ Human input conversion passed
✓ Profile validation passed
✓ Recommendation generation passed
✓ Score validation passed
✓ Reference exclusion passed

✓ COMPLETE USER INPUT PIPELINE PASSED


In [40]:
# =========================================================
# PHASE 9D — EXPLAINABILITY INTEGRATION
# CELL 27 — PRODUCTION DESTINATION EXPLANATION
# =========================================================
#
# This function explains one recommendation using the same
# mathematical components used by the recommendation engine.
#
# Final hybrid score:
#
#     Personalized Score × 0.70
#     +
#     Similarity Score × 0.30
#
# Personalized Score:
#
#     Preference Score × 0.60
#     +
#     Interest Score × 0.40
#
# The explanation therefore reconstructs the final score
# instead of generating a disconnected text explanation.
# =========================================================


def explain_recommendation(
    destination,
    user_preferences,
    user_interests,
    reference_destination=None,
    mode="hybrid"
):
    """
    Explain why a destination received its recommendation score.

    Parameters
    ----------
    destination : str
        Destination to explain.

    user_preferences : dict
        User travel preference weights.

    user_interests : dict
        User interest weights.

    reference_destination : str or None
        Reference destination for similarity mode.

    mode : str
        "preference", "similarity", or "hybrid".

    Returns
    -------
    dict
        Complete mathematical explanation.
    """

    # -----------------------------------------------------
    # Validate destination.
    # -----------------------------------------------------

    if destination not in destinations:

        raise ValueError(
            f"Unknown destination: {destination}"
        )


    # -----------------------------------------------------
    # Calculate the same scores used by the production
    # recommendation engine.
    # -----------------------------------------------------

    preference_scores = (
        calculate_preference_score(
            user_preferences
        )
    )


    interest_scores = (
        calculate_interest_score(
            user_interests
        )
    )


    # -----------------------------------------------------
    # Calculate personalized score.
    # -----------------------------------------------------

    personalized_score = (

        preference_scores
        *
        travel_factor_weight

        +

        interest_scores
        *
        interest_factor_weight
    )


    # -----------------------------------------------------
    # Similarity score.
    # -----------------------------------------------------

    if mode in [
        "similarity",
        "hybrid"
    ]:

        if reference_destination is None:

            raise ValueError(
                "reference_destination is required "
                f"for {mode} mode."
            )


        similarity_scores = (
            calculate_similarity_score(
                reference_destination
            )
        )

    else:

        similarity_scores = pd.Series(
            0.0,
            index=destinations,
            dtype=float
        )


    # -----------------------------------------------------
    # Calculate final score.
    # -----------------------------------------------------

    if mode == "preference":

        final_score = (
            personalized_score
        )

    elif mode == "similarity":

        final_score = (
            similarity_scores
        )

    else:

        final_score = (

            personalized_score
            *
            hybrid_preference_weight

            +

            similarity_scores
            *
            hybrid_similarity_weight
        )


    # -----------------------------------------------------
    # Extract destination-specific values.
    # -----------------------------------------------------

    preference_value = float(
        preference_scores.loc[
            destination
        ]
    )


    interest_value = float(
        interest_scores.loc[
            destination
        ]
    )


    similarity_value = float(
        similarity_scores.loc[
            destination
        ]
    )


    personalized_value = float(
        personalized_score.loc[
            destination
        ]
    )


    final_value = float(
        final_score.loc[
            destination
        ]
    )


    # -----------------------------------------------------
    # Calculate high-level contributions.
    #
    # For hybrid mode:
    #
    # preference contribution
    #     = preference score
    #       × 0.60
    #       × 0.70
    #
    # interest contribution
    #     = interest score
    #       × 0.40
    #       × 0.70
    #
    # similarity contribution
    #     = similarity score
    #       × 0.30
    # -----------------------------------------------------

    if mode == "hybrid":

        preference_contribution = (
            preference_value
            *
            travel_factor_weight
            *
            hybrid_preference_weight
        )


        interest_contribution = (
            interest_value
            *
            interest_factor_weight
            *
            hybrid_preference_weight
        )


        similarity_contribution = (
            similarity_value
            *
            hybrid_similarity_weight
        )


    elif mode == "preference":

        preference_contribution = (
            preference_value
            *
            travel_factor_weight
        )


        interest_contribution = (
            interest_value
            *
            interest_factor_weight
        )


        similarity_contribution = 0.0


    else:

        preference_contribution = 0.0

        interest_contribution = 0.0

        similarity_contribution = (
            similarity_value
        )


    # -----------------------------------------------------
    # Reconstruct final score from contributions.
    # -----------------------------------------------------

    reconstructed_score = (

        preference_contribution

        +

        interest_contribution

        +

        similarity_contribution
    )


    reconstruction_error = abs(
        final_value
        -
        reconstructed_score
    )


    # -----------------------------------------------------
    # Validate mathematical consistency.
    # -----------------------------------------------------

    if reconstruction_error > 1e-10:

        raise ValueError(
            "Explanation reconstruction failed."
        )


    # -----------------------------------------------------
    # Return structured explanation.
    # -----------------------------------------------------

    return {

        "destination":
            destination,

        "mode":
            mode,

        "preference_score":
            preference_value,

        "interest_score":
            interest_value,

        "personalized_score":
            personalized_value,

        "similarity_score":
            similarity_value,

        "final_score":
            final_value,

        "preference_contribution":
            preference_contribution,

        "interest_contribution":
            interest_contribution,

        "similarity_contribution":
            similarity_contribution,

        "reconstructed_score":
            reconstructed_score,

        "reconstruction_error":
            reconstruction_error
    }


print("=" * 60)
print("✓ PRODUCTION EXPLANATION FUNCTION CREATED")
print("=" * 60)

✓ PRODUCTION EXPLANATION FUNCTION CREATED


In [41]:
# =========================================================
# CELL 28 — TEST MUMBAI EXPLANATION
# =========================================================


mumbai_explanation = (
    explain_recommendation(

        destination="Mumbai",

        user_preferences=
            user_preferences,

        user_interests=
            user_interests,

        reference_destination="Goa",

        mode="hybrid"
    )
)


print("=" * 60)
print("WHY WAS MUMBAI RECOMMENDED?")
print("=" * 60)


print(
    "\nDestination:",
    mumbai_explanation[
        "destination"
    ]
)


print(
    "Mode:",
    mumbai_explanation[
        "mode"
    ]
)


print(
    "\nPersonalized preference score:",
    round(
        mumbai_explanation[
            "preference_score"
        ],
        6
    )
)


print(
    "Personalized interest score:",
    round(
        mumbai_explanation[
            "interest_score"
        ],
        6
    )
)


print(
    "Similarity score:",
    round(
        mumbai_explanation[
            "similarity_score"
        ],
        6
    )
)


print(
    "\nFinal recommendation score:",
    round(
        mumbai_explanation[
            "final_score"
        ],
        6
    )
)


print(
    "\n------------------------------------------------------------"
)

print(
    "SCORE CONTRIBUTIONS"
)


print(
    f"\nPreference contribution: "
    f"{mumbai_explanation['preference_contribution']:.6f}"
)


print(
    f"Interest contribution: "
    f"{mumbai_explanation['interest_contribution']:.6f}"
)


print(
    f"Similarity contribution: "
    f"{mumbai_explanation['similarity_contribution']:.6f}"
)


print(
    "\nReconstructed score:",
    f"{mumbai_explanation['reconstructed_score']:.6f}"
)


print(
    "Reconstruction error:",
    f"{mumbai_explanation['reconstruction_error']:.12f}"
)

WHY WAS MUMBAI RECOMMENDED?

Destination: Mumbai
Mode: hybrid

Personalized preference score: 0.32748
Personalized interest score: 0.535142
Similarity score: 0.730195

Final recommendation score: 0.50644

------------------------------------------------------------
SCORE CONTRIBUTIONS

Preference contribution: 0.137542
Interest contribution: 0.149840
Similarity contribution: 0.219059

Reconstructed score: 0.506440
Reconstruction error: 0.000000000000


In [42]:
# =========================================================
# CELL 29 — HUMAN-READABLE RECOMMENDATION EXPLANATION
# =========================================================
#
# This function does NOT change the model score.
#
# It simply converts the already-calculated scores into
# understandable text.
# =========================================================


def format_recommendation_explanation(
    explanation
):
    """
    Convert structured recommendation explanation into
    human-readable text.
    """

    destination = explanation[
        "destination"
    ]

    final_score = explanation[
        "final_score"
    ]

    preference_score = explanation[
        "preference_score"
    ]

    interest_score = explanation[
        "interest_score"
    ]

    similarity_score = explanation[
        "similarity_score"
    ]


    # -----------------------------------------------------
    # Build explanation sentences.
    # -----------------------------------------------------

    text = (

        f"{destination} is recommended with a "
        f"personalized score of "
        f"{final_score:.3f}. "
    )


    text += (

        f"It has a travel-preference score of "
        f"{preference_score:.3f} and an interest "
        f"match score of "
        f"{interest_score:.3f}. "
    )


    # -----------------------------------------------------
    # Add similarity explanation when applicable.
    # -----------------------------------------------------

    if (
        explanation["mode"]
        == "hybrid"
    ):

        text += (

            f"Its similarity to the reference destination "
            f"is {similarity_score:.3f}."
        )


    return text


print("=" * 60)
print("✓ HUMAN-READABLE EXPLANATION FUNCTION CREATED")
print("=" * 60)

✓ HUMAN-READABLE EXPLANATION FUNCTION CREATED


In [43]:
# =========================================================
# CELL 30 — DISPLAY HUMAN-READABLE EXPLANATION
# =========================================================


explanation_text = (
    format_recommendation_explanation(
        mumbai_explanation
    )
)


print("=" * 60)
print("TRAVEL AGENT EXPLANATION")
print("=" * 60)


print(
    "\n" + explanation_text
)

TRAVEL AGENT EXPLANATION

Mumbai is recommended with a personalized score of 0.506. It has a travel-preference score of 0.327 and an interest match score of 0.535. Its similarity to the reference destination is 0.730.


In [44]:
# =========================================================
# CELL 31 — EXPLANATION / RECOMMENDATION CONSISTENCY TEST
# =========================================================
#
# We compare:
#
#     recommendation engine score
#
# against:
#
#     explanation score
#
# They must be mathematically identical.
# =========================================================


print("=" * 60)
print("EXPLANATION CONSISTENCY TEST")
print("=" * 60)


# ---------------------------------------------------------
# Get Mumbai's score from the actual recommendation output.
# ---------------------------------------------------------

mumbai_rows = (
    hybrid_user_recommendations[
        hybrid_user_recommendations[
            "destination"
        ]
        ==
        "Mumbai"
    ]
)


if mumbai_rows.empty:

    raise ValueError(
        "Mumbai was not present in the recommendation output."
    )


engine_score = float(
    mumbai_rows[
        "final_recommendation_score"
    ].iloc[0]
)


# ---------------------------------------------------------
# Get Mumbai's score from the explanation.
# ---------------------------------------------------------

explanation_score = float(
    mumbai_explanation[
        "final_score"
    ]
)


# ---------------------------------------------------------
# Calculate difference.
# ---------------------------------------------------------

score_difference = abs(
    engine_score
    -
    explanation_score
)


print(
    "\nEngine score:",
    f"{engine_score:.12f}"
)


print(
    "Explanation score:",
    f"{explanation_score:.12f}"
)


print(
    "Difference:",
    f"{score_difference:.12f}"
)


# ---------------------------------------------------------
# Validate.
# ---------------------------------------------------------

if score_difference > 1e-10:

    raise ValueError(
        "Explanation score does not match "
        "recommendation engine score."
    )


print(
    "\n✓ Explanation matches recommendation score"
)

print(
    "✓ Mathematical consistency verified"
)

print(
    "\n" + "=" * 60
)

print(
    "✓ EXPLANATION CONSISTENCY TEST PASSED"
)

print(
    "=" * 60
)

EXPLANATION CONSISTENCY TEST

Engine score: 0.506440132010
Explanation score: 0.506440132010
Difference: 0.000000000000

✓ Explanation matches recommendation score
✓ Mathematical consistency verified

✓ EXPLANATION CONSISTENCY TEST PASSED


In [45]:
# =========================================================
# PHASE 9E — COMPONENT-LEVEL EXPLANATION
# CELL 32 — PREFERENCE CONTRIBUTIONS
# =========================================================
#
# This function calculates how much each individual travel
# preference group contributes to a destination's personalized
# preference score.
#
# Groups:
#
#   budget
#   flight
#   accommodation
#   weather
#   destination_characteristics
#
# The contributions are normalized so that they reconstruct
# the destination's personalized preference score.
# =========================================================


def explain_preference_components(
    user_preferences,
    destination
):
    """
    Calculate individual preference-group contributions.

    Returns
    -------
    pandas.DataFrame
        Component-level explanation.
    """

    # -----------------------------------------------------
    # Validate destination.
    # -----------------------------------------------------

    if destination not in destinations:

        raise ValueError(
            f"Unknown destination: {destination}"
        )


    # -----------------------------------------------------
    # Create destination-aligned feature dataset.
    # -----------------------------------------------------

    aligned_features = (
        normalized_features
        .copy()
    )


    aligned_features.index = (
        destinations
    )


    # -----------------------------------------------------
    # Store component results.
    # -----------------------------------------------------

    rows = []


    weighted_total = 0.0

    active_total = 0.0


    # -----------------------------------------------------
    # Process every preference group.
    # -----------------------------------------------------

    for (
        group_name,
        feature_columns
    ) in preference_groups.items():

        user_weight = float(
            user_preferences.get(
                group_name,
                0.0
            )
        )


        # -------------------------------------------------
        # Ignore inactive groups.
        # -------------------------------------------------

        if user_weight <= 0:

            continue


        # -------------------------------------------------
        # Find available features.
        # -------------------------------------------------

        available_columns = [

            column

            for column in feature_columns

            if column
            in aligned_features.columns
        ]


        if not available_columns:

            continue


        # -------------------------------------------------
        # Extract destination feature values.
        # -------------------------------------------------

        values = aligned_features.loc[
            destination,
            available_columns
        ].copy()


        # -------------------------------------------------
        # Apply preference directions.
        # -------------------------------------------------

        for column in available_columns:

            direction = (
                preference_directions.get(
                    column,
                    "higher"
                )
            )


            if direction == "lower":

                values[column] = (
                    1.0
                    -
                    values[column]
                )


        # -------------------------------------------------
        # Calculate the destination's score for this group.
        # -------------------------------------------------

        destination_score = float(
            values.mean()
        )


        if not np.isfinite(
            destination_score
        ):

            destination_score = 0.0


        # -------------------------------------------------
        # Weighted contribution before normalization.
        # -------------------------------------------------

        contribution = (
            destination_score
            *
            user_weight
        )


        weighted_total += contribution

        active_total += user_weight


        rows.append({

            "group":
                group_name,

            "destination_score":
                destination_score,

            "user_weight":
                user_weight,

            "raw_contribution":
                contribution
        })


    # -----------------------------------------------------
    # Handle zero-weight profiles safely.
    # -----------------------------------------------------

    if active_total <= 0:

        return pd.DataFrame(
            columns=[
                "group",
                "destination_score",
                "user_weight",
                "contribution",
                "percentage"
            ]
        )


    # -----------------------------------------------------
    # Normalize raw contributions.
    #
    # This makes the contributions sum to the actual
    # personalized preference score.
    # -----------------------------------------------------

    for row in rows:

        row["contribution"] = (
            row["raw_contribution"]
            /
            active_total
        )


    # -----------------------------------------------------
    # Convert to DataFrame.
    # -----------------------------------------------------

    result = pd.DataFrame(
        rows
    )


    # -----------------------------------------------------
    # Remove temporary column.
    # -----------------------------------------------------

    result = result.drop(
        columns=[
            "raw_contribution"
        ]
    )


    # -----------------------------------------------------
    # Calculate contribution percentages.
    # -----------------------------------------------------

    total_contribution = (
        result["contribution"].sum()
    )


    if total_contribution > 0:

        result["percentage"] = (

            result["contribution"]
            /
            total_contribution
            *
            100
        )

    else:

        result["percentage"] = 0.0


    return result


print("=" * 60)
print("✓ PREFERENCE COMPONENT EXPLANATION CREATED")
print("=" * 60)

✓ PREFERENCE COMPONENT EXPLANATION CREATED


In [50]:
# =========================================================
# PHASE 9E
# CELL 33 — FIXED INTEREST COMPONENT EXPLANATION
# =========================================================
#
# IMPORTANT:
#
# destination_profile_scores has this structure:
#
#   destination
#   nature_score
#   sightseeing_score
#   water_coastal_score
#   wildlife_score
#
# The DataFrame uses RangeIndex:
#
#   0, 1, 2, ..., 49
#
# Therefore, we MUST locate the destination using the
# "destination" column rather than .loc["Mumbai"].
#
# This keeps the explanation aligned with the actual
# production interest-score calculation.
# =========================================================


def explain_interest_components(
    user_interests,
    destination
):
    """
    Calculate individual interest contributions for a
    destination.

    The contributions reconstruct the personalized
    interest score used by the recommendation engine.
    """

    # -----------------------------------------------------
    # STEP 1 — Validate destination.
    # -----------------------------------------------------

    if destination not in destinations:

        raise ValueError(
            f"Unknown destination: {destination}"
        )


    # -----------------------------------------------------
    # STEP 2 — Validate required columns.
    #
    # The saved dataframe contains the destination name
    # as a normal column, not as the index.
    # -----------------------------------------------------

    required_columns = [

        "destination",

        "nature_score",

        "sightseeing_score",

        "water_coastal_score",

        "wildlife_score"
    ]


    missing_columns = [

        column

        for column in required_columns

        if column
        not in destination_profile_scores.columns
    ]


    if missing_columns:

        raise ValueError(
            "Missing interest profile columns: "
            f"{missing_columns}"
        )


    # -----------------------------------------------------
    # STEP 3 — Find the destination row.
    # -----------------------------------------------------

    destination_rows = (
        destination_profile_scores[
            destination_profile_scores[
                "destination"
            ]
            ==
            destination
        ]
    )


    if destination_rows.empty:

        raise ValueError(
            f"Destination '{destination}' "
            "not found in interest profiles."
        )


    if len(destination_rows) > 1:

        raise ValueError(
            f"Duplicate interest profile found "
            f"for destination '{destination}'."
        )


    destination_row = (
        destination_rows.iloc[0]
    )


    # -----------------------------------------------------
    # STEP 4 — Map interest names to the actual dataframe
    # column names.
    # -----------------------------------------------------

    interest_columns = {

        "nature":
            "nature_score",

        "sightseeing":
            "sightseeing_score",

        "water_coastal":
            "water_coastal_score",

        "wildlife":
            "wildlife_score"
    }


    # -----------------------------------------------------
    # STEP 5 — Calculate weighted contributions.
    # -----------------------------------------------------

    rows = []

    active_total = 0.0


    for interest in interest_profiles:

        # -------------------------------------------------
        # Get the user's interest weight.
        # -------------------------------------------------

        user_weight = float(
            user_interests.get(
                interest,
                0.0
            )
        )


        # -------------------------------------------------
        # Ignore inactive interests.
        # -------------------------------------------------

        if user_weight <= 0:

            continue


        # -------------------------------------------------
        # Find corresponding profile column.
        # -------------------------------------------------

        score_column = (
            interest_columns[
                interest
            ]
        )


        # -------------------------------------------------
        # Get destination's interest score.
        # -------------------------------------------------

        destination_score = float(
            destination_row[
                score_column
            ]
        )


        # -------------------------------------------------
        # Protect against invalid values.
        # -------------------------------------------------

        if not np.isfinite(
            destination_score
        ):

            destination_score = 0.0


        # -------------------------------------------------
        # Weighted contribution before normalization.
        # -------------------------------------------------

        raw_contribution = (
            destination_score
            *
            user_weight
        )


        active_total += user_weight


        rows.append({

            "interest":
                interest,

            "destination_score":
                destination_score,

            "user_weight":
                user_weight,

            "raw_contribution":
                raw_contribution
        })


    # -----------------------------------------------------
    # STEP 6 — Zero-interest safety.
    # -----------------------------------------------------

    if active_total <= 0:

        return pd.DataFrame(
            columns=[

                "interest",

                "destination_score",

                "user_weight",

                "contribution",

                "percentage"
            ]
        )


    # -----------------------------------------------------
    # STEP 7 — Normalize contributions.
    #
    # This produces the same weighted-average calculation
    # used by the production interest scorer.
    # -----------------------------------------------------

    for row in rows:

        row["contribution"] = (

            row["raw_contribution"]

            /

            active_total
        )


    # -----------------------------------------------------
    # STEP 8 — Convert to DataFrame.
    # -----------------------------------------------------

    result = pd.DataFrame(
        rows
    )


    # -----------------------------------------------------
    # Remove temporary raw contribution.
    # -----------------------------------------------------

    result = result.drop(
        columns=[
            "raw_contribution"
        ]
    )


    # -----------------------------------------------------
    # STEP 9 — Calculate percentage contribution.
    # -----------------------------------------------------

    total_contribution = (
        result[
            "contribution"
        ]
        .sum()
    )


    if total_contribution > 0:

        result["percentage"] = (

            result[
                "contribution"
            ]

            /

            total_contribution

            *

            100.0
        )

    else:

        result["percentage"] = 0.0


    return result


print("=" * 60)
print("✓ FIXED INTEREST COMPONENT EXPLANATION CREATED")
print("=" * 60)

✓ FIXED INTEREST COMPONENT EXPLANATION CREATED


In [51]:
# =========================================================
# CELL 34A — VERIFY MUMBAI INTEREST COMPONENTS
# =========================================================


print("=" * 60)
print("MUMBAI INTEREST COMPONENT CHECK")
print("=" * 60)


mumbai_interest_components = (
    explain_interest_components(

        user_interests=
            user_interests,

        destination="Mumbai"
    )
)


print(
    "\nMumbai interest components:"
)


print(
    mumbai_interest_components
    .to_string(
        index=False
    )
)


print(
    "\nComponent contribution total:"
)


print(
    mumbai_interest_components[
        "contribution"
    ].sum()
)


# ---------------------------------------------------------
# Compare with production score.
# ---------------------------------------------------------

mumbai_interest_production = float(
    calculate_interest_score(
        user_interests
    )
    .loc[
        "Mumbai"
    ]
)


print(
    "\nProduction interest score:"
)


print(
    mumbai_interest_production
)


print(
    "\nDifference:"
)


print(
    abs(
        mumbai_interest_components[
            "contribution"
        ].sum()
        -
        mumbai_interest_production
    )
)

MUMBAI INTEREST COMPONENT CHECK

Mumbai interest components:
     interest  destination_score  user_weight  contribution  percentage
       nature           0.437710          1.0      0.182379   34.080472
  sightseeing           0.829316          0.6      0.207329   38.742803
water_coastal           0.149660          0.2      0.012472    2.330531
     wildlife           0.531850          0.6      0.132962   24.846194

Component contribution total:
0.5351421312482234

Production interest score:
0.5351421312482234

Difference:
0.0


In [52]:
# =========================================================
# CELL 34 — TEST COMPONENT EXPLANATION
# =========================================================


mumbai_preference_components = (
    explain_preference_components(

        user_preferences=
            user_preferences,

        destination="Mumbai"
    )
)


mumbai_interest_components = (
    explain_interest_components(

        user_interests=
            user_interests,

        destination="Mumbai"
    )
)


print("=" * 60)
print("MUMBAI — PREFERENCE COMPONENTS")
print("=" * 60)


print(
    mumbai_preference_components
    .to_string(
        index=False
    )
)


print(
    "\n" + "=" * 60
)

print(
    "MUMBAI — INTEREST COMPONENTS"
)

print(
    "=" * 60
)


print(
    mumbai_interest_components
    .to_string(
        index=False
    )
)

MUMBAI — PREFERENCE COMPONENTS
                      group  destination_score  user_weight  contribution  percentage
                     budget           0.078161          1.0      0.020569    6.280855
                     flight           0.178504          0.6      0.028185    8.606575
              accommodation           0.330886          0.6      0.052245   15.953692
                    weather           0.564178          0.8      0.118774   36.269119
destination_characteristics           0.511611          0.8      0.107707   32.889759

MUMBAI — INTEREST COMPONENTS
     interest  destination_score  user_weight  contribution  percentage
       nature           0.437710          1.0      0.182379   34.080472
  sightseeing           0.829316          0.6      0.207329   38.742803
water_coastal           0.149660          0.2      0.012472    2.330531
     wildlife           0.531850          0.6      0.132962   24.846194


In [53]:
# =========================================================
# CELL 35 — COMPONENT RECONSTRUCTION VALIDATION
# =========================================================
#
# The individual components must reconstruct:
#
#     personalized preference score
#
# and
#
#     personalized interest score
#
# exactly.
# =========================================================


print("=" * 60)
print("COMPONENT RECONSTRUCTION VALIDATION")
print("=" * 60)


# ---------------------------------------------------------
# Calculate production scores.
# ---------------------------------------------------------

production_preference = float(
    calculate_preference_score(
        user_preferences
    ).loc[
        "Mumbai"
    ]
)


production_interest = float(
    calculate_interest_score(
        user_interests
    ).loc[
        "Mumbai"
    ]
)


# ---------------------------------------------------------
# Reconstruct preference score.
# ---------------------------------------------------------

reconstructed_preference = float(
    mumbai_preference_components[
        "contribution"
    ].sum()
)


# ---------------------------------------------------------
# Reconstruct interest score.
# ---------------------------------------------------------

reconstructed_interest = float(
    mumbai_interest_components[
        "contribution"
    ].sum()
)


# ---------------------------------------------------------
# Calculate errors.
# ---------------------------------------------------------

preference_error = abs(
    production_preference
    -
    reconstructed_preference
)


interest_error = abs(
    production_interest
    -
    reconstructed_interest
)


print(
    "\nPreference production score:",
    f"{production_preference:.12f}"
)


print(
    "Preference reconstructed score:",
    f"{reconstructed_preference:.12f}"
)


print(
    "Preference reconstruction error:",
    f"{preference_error:.12f}"
)


print(
    "\nInterest production score:",
    f"{production_interest:.12f}"
)


print(
    "Interest reconstructed score:",
    f"{reconstructed_interest:.12f}"
)


print(
    "Interest reconstruction error:",
    f"{interest_error:.12f}"
)


# ---------------------------------------------------------
# Validate.
# ---------------------------------------------------------

if preference_error > 1e-10:

    raise ValueError(
        "Preference components do not reconstruct "
        "the production score."
    )


if interest_error > 1e-10:

    raise ValueError(
        "Interest components do not reconstruct "
        "the production score."
    )


print(
    "\n✓ Preference components reconstruct "
    "production score."
)


print(
    "✓ Interest components reconstruct "
    "production score."
)


print(
    "\n" + "=" * 60
)

print(
    "✓ COMPONENT RECONSTRUCTION VALIDATION PASSED"
)

print(
    "=" * 60
)

COMPONENT RECONSTRUCTION VALIDATION

Preference production score: 0.327480296683
Preference reconstructed score: 0.327480296683
Preference reconstruction error: 0.000000000000

Interest production score: 0.535142131248
Interest reconstructed score: 0.535142131248
Interest reconstruction error: 0.000000000000

✓ Preference components reconstruct production score.
✓ Interest components reconstruct production score.

✓ COMPONENT RECONSTRUCTION VALIDATION PASSED


In [54]:
# =========================================================
# PHASE 9F — COMPLETE TRAVEL AGENT
# CELL 36 — UNIFIED TRAVEL AGENT FUNCTION
# =========================================================
#
# This function combines everything we have built so far:
#
#   1. Human-readable preferences
#   2. Human-readable interests
#   3. Input validation
#   4. User profile construction
#   5. Recommendation engine
#   6. Component-level explanation
#   7. Similarity explanation
#
# IMPORTANT:
#
# This function does NOT implement a new scoring algorithm.
#
# It simply connects the already-tested components into
# one production-level interface.
# =========================================================


def travel_agent(
    budget="Medium",
    flight="Medium",
    accommodation="Medium",
    weather="Medium",
    destination_characteristics="Medium",
    nature="Medium",
    sightseeing="Medium",
    water_coastal="Medium",
    wildlife="Medium",
    reference_destination=None,
    mode="hybrid",
    top_k=5
):
    """
    Complete Travel Agent recommendation pipeline.

    Parameters
    ----------
    budget : str
        Importance of budget.

    flight : str
        Importance of flight convenience.

    accommodation : str
        Importance of accommodation.

    weather : str
        Importance of weather.

    destination_characteristics : str
        Importance of destination characteristics.

    nature : str
        Interest in nature.

    sightseeing : str
        Interest in sightseeing.

    water_coastal : str
        Interest in water/coastal destinations.

    wildlife : str
        Interest in wildlife.

    reference_destination : str or None
        Optional destination to find alternatives to.

    mode : str
        "preference", "similarity", or "hybrid".

    top_k : int
        Number of recommendations.

    Returns
    -------
    dict
        User profile, recommendations, and explanations.
    """

    # -----------------------------------------------------
    # STEP 1 — Validate recommendation mode.
    # -----------------------------------------------------

    valid_modes = [

        "preference",

        "similarity",

        "hybrid"
    ]


    if mode not in valid_modes:

        raise ValueError(
            f"Invalid mode '{mode}'. "
            f"Choose from {valid_modes}."
        )


    # -----------------------------------------------------
    # STEP 2 — Validate Top-K.
    # -----------------------------------------------------

    if not isinstance(
        top_k,
        int
    ) or top_k <= 0:

        raise ValueError(
            "top_k must be a positive integer."
        )


    # -----------------------------------------------------
    # STEP 3 — Build numerical user profile from
    # human-readable inputs.
    # -----------------------------------------------------

    user_preferences, user_interests = (
        build_user_profile(

            budget=budget,

            flight=flight,

            accommodation=accommodation,

            weather=weather,

            destination_characteristics=
                destination_characteristics,

            nature=nature,

            sightseeing=sightseeing,

            water_coastal=water_coastal,

            wildlife=wildlife
        )
    )


    # -----------------------------------------------------
    # STEP 4 — Similarity/hybrid modes require a reference
    # destination.
    # -----------------------------------------------------

    if mode in [

        "similarity",

        "hybrid"

    ]:

        if reference_destination is None:

            raise ValueError(
                f"reference_destination is required "
                f"for {mode} mode."
            )


        if reference_destination not in destinations:

            raise ValueError(
                f"Unknown reference destination: "
                f"{reference_destination}"
            )


    # -----------------------------------------------------
    # STEP 5 — Generate recommendations using the
    # already-tested production engine.
    # -----------------------------------------------------

    recommendations = (
        get_recommendations(

            user_preferences=
                user_preferences,

            user_interests=
                user_interests,

            reference_destination=
                reference_destination,

            mode=mode,

            top_k=top_k
        )
    )


    # -----------------------------------------------------
    # STEP 6 — Generate detailed explanation for every
    # returned destination.
    # -----------------------------------------------------

    explanations = {}


    for destination in (
        recommendations[
            "destination"
        ]
        .tolist()
    ):

        explanation = (
            explain_recommendation(

                destination=
                    destination,

                user_preferences=
                    user_preferences,

                user_interests=
                    user_interests,

                reference_destination=
                    reference_destination,

                mode=mode
            )
        )


        # -------------------------------------------------
        # Add individual preference components.
        # -------------------------------------------------

        explanation[
            "preference_components"
        ] = (
            explain_preference_components(

                user_preferences=
                    user_preferences,

                destination=
                    destination
            )
        )


        # -------------------------------------------------
        # Add individual interest components.
        # -------------------------------------------------

        explanation[
            "interest_components"
        ] = (
            explain_interest_components(

                user_interests=
                    user_interests,

                destination=
                    destination
            )
        )


        explanations[
            destination
        ] = explanation


    # -----------------------------------------------------
    # STEP 7 — Return everything required by the
    # application layer.
    # -----------------------------------------------------

    return {

        "user_preferences":
            user_preferences,

        "user_interests":
            user_interests,

        "reference_destination":
            reference_destination,

        "mode":
            mode,

        "recommendations":
            recommendations,

        "explanations":
            explanations
    }


print("=" * 60)
print("✓ COMPLETE TRAVEL AGENT FUNCTION CREATED")
print("=" * 60)

✓ COMPLETE TRAVEL AGENT FUNCTION CREATED


In [55]:
# =========================================================
# CELL 37 — COMPLETE TRAVEL AGENT TEST
# =========================================================


travel_result = (
    travel_agent(

        budget="Very High",

        flight="Medium",

        accommodation="Medium",

        weather="High",

        destination_characteristics="High",

        nature="Very High",

        sightseeing="Medium",

        water_coastal="Low",

        wildlife="Medium",

        reference_destination="Goa",

        mode="hybrid",

        top_k=5
    )
)


print("=" * 60)
print("COMPLETE TRAVEL AGENT RESULT")
print("=" * 60)


print(
    "\nUSER PREFERENCES"
)

for group, weight in (
    travel_result[
        "user_preferences"
    ].items()
):

    print(
        f"  {group:<32} {weight}"
    )


print(
    "\nUSER INTERESTS"
)

for interest, weight in (
    travel_result[
        "user_interests"
    ].items()
):

    print(
        f"  {interest:<32} {weight}"
    )


print(
    "\nREFERENCE DESTINATION:",
    travel_result[
        "reference_destination"
    ]
)


print(
    "\nRECOMMENDATIONS"
)


print(
    travel_result[
        "recommendations"
    ][
        [
            "rank",

            "destination",

            "personalized_preference_score",

            "personalized_interest_score",

            "similarity_score",

            "final_recommendation_score"
        ]
    ]
    .to_string(
        index=False
    )
)

COMPLETE TRAVEL AGENT RESULT

USER PREFERENCES
  budget                           1.0
  flight                           0.6
  accommodation                    0.6
  weather                          0.8
  destination_characteristics      0.8

USER INTERESTS
  nature                           1.0
  sightseeing                      0.6
  water_coastal                    0.2
  wildlife                         0.6

REFERENCE DESTINATION: Goa

RECOMMENDATIONS
 rank destination  personalized_preference_score  personalized_interest_score  similarity_score  final_recommendation_score
    1      Mumbai                       0.327480                     0.535142          0.730195                    0.506440
    2       Delhi                       0.378230                     0.424713          0.478832                    0.421426
    3      Jaipur                       0.312487                     0.418880          0.460086                    0.386557
    4    Srinagar                       0.386

In [56]:
# =========================================================
# CELL 38 — COMPLETE TRAVEL AGENT VALIDATION
# =========================================================
#
# This is an integration test.
#
# It verifies that all major components work together.
# =========================================================


print("=" * 60)
print("COMPLETE TRAVEL AGENT VALIDATION")
print("=" * 60)


recommendations = (
    travel_result[
        "recommendations"
    ]
)


explanations = (
    travel_result[
        "explanations"
    ]
)


# ---------------------------------------------------------
# Check recommendation count.
# ---------------------------------------------------------

if len(
    recommendations
) != 5:

    raise ValueError(
        "Incorrect number of recommendations."
    )


# ---------------------------------------------------------
# Check destination uniqueness.
# ---------------------------------------------------------

if (
    recommendations[
        "destination"
    ]
    .nunique()
    !=
    len(recommendations)
):

    raise ValueError(
        "Duplicate destinations detected."
    )


# ---------------------------------------------------------
# Check Goa exclusion.
# ---------------------------------------------------------

if (
    "Goa"
    in recommendations[
        "destination"
    ].values
):

    raise ValueError(
        "Reference destination Goa was not excluded."
    )


# ---------------------------------------------------------
# Validate every recommendation.
# ---------------------------------------------------------

for _, row in (
    recommendations.iterrows()
):

    destination = row[
        "destination"
    ]


    # -----------------------------------------------------
    # Explanation must exist.
    # -----------------------------------------------------

    if destination not in explanations:

        raise ValueError(
            f"Missing explanation for "
            f"{destination}."
        )


    explanation = (
        explanations[
            destination
        ]
    )


    # -----------------------------------------------------
    # Compare recommendation score and explanation score.
    # -----------------------------------------------------

    engine_score = float(
        row[
            "final_recommendation_score"
        ]
    )


    explanation_score = float(
        explanation[
            "final_score"
        ]
    )


    score_error = abs(
        engine_score
        -
        explanation_score
    )


    if score_error > 1e-10:

        raise ValueError(
            f"Explanation mismatch for "
            f"{destination}."
        )


    # -----------------------------------------------------
    # Verify explanation reconstruction.
    # -----------------------------------------------------

    reconstruction_error = (
        explanation[
            "reconstruction_error"
        ]
    )


    if reconstruction_error > 1e-10:

        raise ValueError(
            f"Reconstruction failed for "
            f"{destination}."
        )


# ---------------------------------------------------------
# Final success message.
# ---------------------------------------------------------

print(
    "\n✓ Recommendation count valid"
)

print(
    "✓ Destinations are unique"
)

print(
    "✓ Reference destination excluded"
)

print(
    "✓ Every recommendation has an explanation"
)

print(
    "✓ Explanation scores match engine scores"
)

print(
    "✓ Mathematical reconstruction validated"
)


print(
    "\n" + "=" * 60
)

print(
    "✓ COMPLETE TRAVEL AGENT VALIDATION PASSED"
)

print(
    "=" * 60
)

COMPLETE TRAVEL AGENT VALIDATION

✓ Recommendation count valid
✓ Destinations are unique
✓ Reference destination excluded
✓ Every recommendation has an explanation
✓ Explanation scores match engine scores
✓ Mathematical reconstruction validated

✓ COMPLETE TRAVEL AGENT VALIDATION PASSED
